# Notebook 3 — Frequency-Support Exploration

## Research questions

1. What information is encoded in `frequency_support`?
2. Is its grammar stable across Archive records?
3. At what level is it stable: Member, source, ASDM, or row?
4. Does each parsed interval correspond to one logical SPW?
5. How do parsed intervals relate to row-level `frequency` and `bandwidth`?
6. Which resolution, sensitivity, and polarization formats occur?
7. Which cases cannot be parsed safely?

## Step 1 — Environment, TAP Connection, and Schema Validation

This step verifies the Python environment, connects to the European ALMA TAP
service, and checks whether the fields required by this investigation are
available in `ivoa.obscore`.

The Notebook does not assume that every requested field exists. Missing fields
are reported before the scientific queries are constructed.

In [1]:
import re
import sys
import time
from collections import Counter
from typing import Any

import astropy
import astropy.units as u
import numpy as np
import pandas as pd
import pyvo

from IPython.display import display


pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 180)
pd.set_option("display.width", 220)


print("Python executable:", sys.executable)
print("Python version:", sys.version.split()[0])
print("PyVO version:", pyvo.__version__)
print("Astropy version:", astropy.__version__)
print("Pandas version:", pd.__version__)

Python executable: /Users/nana/opt/anaconda3/envs/alma-duplication/bin/python
Python version: 3.12.11
PyVO version: 1.9.1
Astropy version: 8.0.1
Pandas version: 3.0.5


In [2]:
TAP_URL = "https://almascience.eso.org/tap"

service = pyvo.dal.TAPService(TAP_URL)

print("TAP service created:")
print(service.baseurl)

TAP service created:
https://almascience.eso.org/tap


In [3]:
def run_tap_query(
    query: str,
    maxrec: int = 1000,
    label: str | None = None,
):
    """Execute an ADQL query and return an Astropy Table."""

    if label:
        print(f"Running: {label}")

    started_at = time.perf_counter()

    response = service.search(
        query,
        maxrec=maxrec,
    )

    table = response.to_table()

    elapsed_seconds = time.perf_counter() - started_at

    print(
        f"Retrieved {len(table):,} rows "
        f"in {elapsed_seconds:.1f} seconds."
    )

    return table

In [4]:
requested_archive_columns = [
    "proposal_id",
    "group_ous_uid",
    "member_ous_uid",
    "asdm_uid",
    "obs_id",
    "target_name",
    "s_ra",
    "s_dec",
    "s_region",
    "frequency",
    "bandwidth",
    "frequency_support",
    "spatial_resolution",
    "sensitivity_10kms",
    "cont_sensitivity_bandwidth",
    "antenna_arrays",
    "is_mosaic",
    "t_min",
    "t_max",
    "obs_release_date",
]


schema_query = """
SELECT
    column_name,
    datatype,
    unit,
    description
FROM TAP_SCHEMA.columns
WHERE table_name = 'ivoa.obscore'
"""


schema_table = run_tap_query(
    schema_query,
    maxrec=1000,
    label="Read ivoa.obscore schema",
)

schema_df = schema_table.to_pandas()

schema_df["column_name"] = (
    schema_df["column_name"]
    .astype(str)
    .str.strip()
)

available_columns = set(schema_df["column_name"])

selected_archive_columns = [
    column_name
    for column_name in requested_archive_columns
    if column_name in available_columns
]

missing_requested_columns = [
    column_name
    for column_name in requested_archive_columns
    if column_name not in available_columns
]


print("Selected columns:")
print(selected_archive_columns)

print("\nRequested columns not available:")
print(missing_requested_columns)

Running: Read ivoa.obscore schema
Retrieved 73 rows in 1.5 seconds.
Selected columns:
['proposal_id', 'group_ous_uid', 'member_ous_uid', 'asdm_uid', 'obs_id', 'target_name', 's_ra', 's_dec', 's_region', 'frequency', 'bandwidth', 'frequency_support', 'spatial_resolution', 'sensitivity_10kms', 'cont_sensitivity_bandwidth', 'antenna_arrays', 'is_mosaic', 't_min', 't_max', 'obs_release_date']

Requested columns not available:
[]


In [5]:
selected_schema_df = (
    schema_df[
        schema_df["column_name"].isin(
            requested_archive_columns
        )
    ]
    .sort_values("column_name")
    .reset_index(drop=True)
)

display(selected_schema_df)

,column_name,datatype,unit,description
0,antenna_arrays,char,,"Blank-separated list of Pad:Antenna pairs, i.e., A109:DV09 J504:DV02 J505:DV05 for antennas DV09, DV02 and DV05 sitting on pads A109, J504, and J505, respectively."
1,asdm_uid,char,,UID of the ASDM containing this Field.
2,bandwidth,double,Hz,Total Bandwidth
3,cont_sensitivity_bandwidth,double,mJy/beam,"Estimated noise in the aggregated continuum bandwidth. Note this is an indication only, it does not include the effects of flagging or dynamic range limitations."
4,frequency,double,GHz,Observed (tuned) reference frequency on the sky.
5,frequency_support,char,GHz,All frequency ranges used by the field
6,group_ous_uid,char,,Group OUS ID
7,is_mosaic,char,,Flag to indicate if this ASDM represents a mosaic or not.
8,member_ous_uid,char,,Member OUS ID
9,obs_id,char,,internal dataset identifier


## Step 2 — Construct a Purposive Member OUS Sample

The sample is designed to expose different frequency-support structures rather
than estimate Archive-wide frequencies.

The selection includes:

- low-, medium-, and high-frequency records;
- mosaic and non-mosaic records;
- recent proposals;
- Member OUS datasets that can later be classified as single-source,
  multi-source, single-ASDM, or multi-ASDM.

The selected Member OUS identifiers form the sample manifest for the remaining
experiments.

In [6]:
sample_strata = [
    {
        "label": "low_frequency_non_mosaic",
        "frequency_min_ghz": 84.0,
        "frequency_max_ghz": 116.0,
        "is_mosaic": "F",
        "proposal_prefix": None,
    },
    {
        "label": "mid_frequency_non_mosaic",
        "frequency_min_ghz": 211.0,
        "frequency_max_ghz": 275.0,
        "is_mosaic": "F",
        "proposal_prefix": None,
    },
    {
        "label": "high_frequency_non_mosaic",
        "frequency_min_ghz": 385.0,
        "frequency_max_ghz": 500.0,
        "is_mosaic": "F",
        "proposal_prefix": None,
    },
    {
        "label": "low_frequency_mosaic",
        "frequency_min_ghz": 84.0,
        "frequency_max_ghz": 116.0,
        "is_mosaic": "T",
        "proposal_prefix": None,
    },
    {
        "label": "mid_frequency_mosaic",
        "frequency_min_ghz": 211.0,
        "frequency_max_ghz": 275.0,
        "is_mosaic": "T",
        "proposal_prefix": None,
    },
    {
        "label": "recent_mid_frequency",
        "frequency_min_ghz": 211.0,
        "frequency_max_ghz": 275.0,
        "is_mosaic": None,
        "proposal_prefix": "202",
    },
]

In [7]:
candidate_member_tables = []

for stratum in sample_strata:
    conditions = [
        "science_observation = 'T'",
        "member_ous_uid IS NOT NULL",
        (
            f"frequency >= "
            f"{stratum['frequency_min_ghz']}"
        ),
        (
            f"frequency < "
            f"{stratum['frequency_max_ghz']}"
        ),
    ]

    if stratum["is_mosaic"] is not None:
        conditions.append(
            f"is_mosaic = '{stratum['is_mosaic']}'"
        )

    if stratum["proposal_prefix"] is not None:
        conditions.append(
            "proposal_id LIKE "
            f"'{stratum['proposal_prefix']}%'"
        )

    where_clause = "\nAND ".join(conditions)

    candidate_query = f"""
SELECT DISTINCT TOP 3
    member_ous_uid,
    proposal_id,
    is_mosaic
FROM ivoa.obscore
WHERE {where_clause}
"""

    candidate_table = run_tap_query(
        candidate_query,
        maxrec=20,
        label=stratum["label"],
    )

    candidate_df = candidate_table.to_pandas()

    if candidate_df.empty:
        print(
            f"No candidates found for "
            f"{stratum['label']}."
        )
        continue

    candidate_df["sample_stratum"] = stratum["label"]
    candidate_df["frequency_min_ghz"] = (
        stratum["frequency_min_ghz"]
    )
    candidate_df["frequency_max_ghz"] = (
        stratum["frequency_max_ghz"]
    )

    candidate_member_tables.append(candidate_df)

Running: low_frequency_non_mosaic
Retrieved 3 rows in 21.6 seconds.
Running: mid_frequency_non_mosaic
Retrieved 3 rows in 5.4 seconds.
Running: high_frequency_non_mosaic
Retrieved 3 rows in 4.8 seconds.
Running: low_frequency_mosaic
Retrieved 3 rows in 5.2 seconds.
Running: mid_frequency_mosaic
Retrieved 3 rows in 2.1 seconds.
Running: recent_mid_frequency
Retrieved 3 rows in 9.5 seconds.


In [8]:
if not candidate_member_tables:
    raise RuntimeError(
        "No Member OUS candidates were returned."
    )


candidate_member_df = pd.concat(
    candidate_member_tables,
    ignore_index=True,
)


candidate_member_df["member_ous_uid"] = (
    candidate_member_df["member_ous_uid"]
    .astype(str)
    .str.strip()
)


candidate_member_summary_df = (
    candidate_member_df
    .groupby(
        "member_ous_uid",
        dropna=False,
    )
    .agg(
        proposal_id=("proposal_id", "first"),
        is_mosaic=("is_mosaic", "first"),
        sample_strata=(
            "sample_stratum",
            lambda values: " | ".join(
                sorted(set(map(str, values)))
            ),
        ),
    )
    .reset_index()
)


print(
    "Unique Member OUS candidates:",
    candidate_member_summary_df[
        "member_ous_uid"
    ].nunique(),
)

display(candidate_member_summary_df)

Unique Member OUS candidates: 18


,member_ous_uid,proposal_id,is_mosaic,sample_strata
0,uid://A001/X11d/Xf,2013.1.00034.S,F,mid_frequency_non_mosaic
1,uid://A001/X11e/X5,2013.1.00088.S,F,mid_frequency_non_mosaic
2,uid://A001/X11f/X6,2013.1.00518.S,T,mid_frequency_mosaic
3,uid://A001/X11f/X8,2013.1.00518.S,T,mid_frequency_mosaic
4,uid://A001/X123/Xc,2013.1.01230.S,F,high_frequency_non_mosaic
5,uid://A001/X12a/X9,2013.1.00907.S,F,high_frequency_non_mosaic
6,uid://A001/X12d/X6,2013.1.00266.S,F,low_frequency_non_mosaic
7,uid://A001/X12d/Xa,2013.1.00266.S,F,low_frequency_non_mosaic
8,uid://A001/X132/Xa,2013.1.00833.S,F,low_frequency_non_mosaic
9,uid://A001/X137/Xd,2013.1.00048.S,F,high_frequency_non_mosaic


## Step 3 — Retrieve Complete Member OUS Records

The candidate query selected Member OUS identifiers rather than individual
Archive rows. This step retrieves every science-observation row associated
with those identifiers.

A separate count query is executed first so that the Notebook does not silently
accept a truncated response.

In [9]:
member_ous_uids = (
    candidate_member_summary_df[
        "member_ous_uid"
    ]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)


def quote_adql_string(value: str) -> str:
    """Quote one string literal for the controlled ADQL sample."""

    escaped_value = value.replace("'", "''")
    return f"'{escaped_value}'"


member_uid_sql = ",\n    ".join(
    quote_adql_string(member_uid)
    for member_uid in member_ous_uids
)


print("Member OUS identifiers selected:")
print(len(member_ous_uids))

Member OUS identifiers selected:
18


In [12]:
count_query = f"""
SELECT COUNT(*) AS total_rows
FROM ivoa.obscore
WHERE science_observation = 'T'
AND member_ous_uid IN (
    {member_uid_sql}
)
"""


count_table = run_tap_query(
    count_query,
    maxrec=10,
    label="Count complete Member OUS rows",
)


expected_total_rows = int(
    count_table["total_rows"][0]
)


print(
    "Expected complete row count:",
    expected_total_rows,
)

MAX_EXPLORATORY_ROWS = 20_000

if expected_total_rows > MAX_EXPLORATORY_ROWS:
    raise RuntimeError(
        "The selected sample contains "
        f"{expected_total_rows:,} rows, which exceeds "
        f"the exploratory limit of "
        f"{MAX_EXPLORATORY_ROWS:,}. "
        "Reduce the number of candidate Member OUS datasets."
    )

Running: Count complete Member OUS rows
Retrieved 1 rows in 1.0 seconds.
Expected complete row count: 171


In [13]:
column_sql = ",\n    ".join(
    selected_archive_columns
)


complete_sample_query = f"""
SELECT
    {column_sql}
FROM ivoa.obscore
WHERE science_observation = 'T'
AND member_ous_uid IN (
    {member_uid_sql}
)
"""


archive_sample_table = run_tap_query(
    complete_sample_query,
    maxrec=max(expected_total_rows, 1),
    label="Retrieve complete frequency-support sample",
)


archive_sample_df = (
    archive_sample_table
    .to_pandas()
    .copy()
)


print("Expected rows:", expected_total_rows)
print("Retrieved rows:", len(archive_sample_df))
print(
    "Complete retrieval:",
    len(archive_sample_df) == expected_total_rows,
)

Running: Retrieve complete frequency-support sample
Retrieved 171 rows in 2.1 seconds.
Expected rows: 171
Retrieved rows: 171
Complete retrieval: True


In [14]:
archive_unit_records = []

for column_name in archive_sample_table.colnames:
    column = archive_sample_table[column_name]

    archive_unit_records.append(
        {
            "column_name": column_name,
            "dtype": str(column.dtype),
            "unit": (
                str(column.unit)
                if column.unit is not None
                else None
            ),
        }
    )


archive_units_df = pd.DataFrame(
    archive_unit_records
)


display(
    archive_units_df[
        archive_units_df["column_name"].isin(
            [
                "frequency",
                "bandwidth",
                "spatial_resolution",
                "sensitivity_10kms",
                "cont_sensitivity_bandwidth",
                "t_min",
                "t_max",
            ]
        )
    ]
)

,column_name,dtype,unit
9,frequency,float64,GHz
10,bandwidth,float64,Hz
12,spatial_resolution,float64,arcsec
13,sensitivity_10kms,float64,mJy / beam
14,cont_sensitivity_bandwidth,float64,mJy / beam
17,t_min,float64,d
18,t_max,float64,d


In [15]:
OBS_ID_PATTERN = re.compile(
    r"\.source\.(?P<source>.*?)"
    r"\.spw\.(?P<spw>[^.]+)$"
)


def parse_obs_id(value: Any) -> tuple[str | None, str | None]:
    if pd.isna(value):
        return None, None

    value_text = str(value).strip()
    match = OBS_ID_PATTERN.search(value_text)

    if match is None:
        return None, None

    return (
        match.group("source"),
        match.group("spw"),
    )


parsed_obs_ids = archive_sample_df["obs_id"].map(
    parse_obs_id
)


archive_sample_df["obs_id_source"] = [
    parsed_value[0]
    for parsed_value in parsed_obs_ids
]

archive_sample_df["spw_identifier"] = [
    parsed_value[1]
    for parsed_value in parsed_obs_ids
]


print(
    "Rows with unparsed obs_id:",
    archive_sample_df[
        "obs_id_source"
    ].isna().sum(),
)

Rows with unparsed obs_id: 0


In [16]:
archive_sample_df = archive_sample_df.merge(
    candidate_member_summary_df[
        [
            "member_ous_uid",
            "sample_strata",
        ]
    ],
    on="member_ous_uid",
    how="left",
    validate="many_to_one",
)

In [17]:
def count_missing_or_blank(
    values: pd.Series,
) -> int:
    as_text = values.astype("string")

    return int(
        (
            as_text.isna()
            | as_text.str.strip().eq("")
        ).sum()
    )


sample_manifest_df = (
    archive_sample_df
    .groupby(
        "member_ous_uid",
        dropna=False,
    )
    .agg(
        proposal_id=("proposal_id", "first"),
        sample_strata=("sample_strata", "first"),
        archive_rows=("obs_id", "size"),
        source_count=("obs_id_source", "nunique"),
        spw_count=("spw_identifier", "nunique"),
        asdm_count=("asdm_uid", "nunique"),
        target_name_count=("target_name", "nunique"),
        frequency_min_ghz=("frequency", "min"),
        frequency_max_ghz=("frequency", "max"),
        mosaic_state_count=("is_mosaic", "nunique"),
        mosaic_state=("is_mosaic", "first"),
        frequency_support_signature_count=(
            "frequency_support",
            "nunique",
        ),
        frequency_support_missing_count=(
            "frequency_support",
            count_missing_or_blank,
        ),
    )
    .reset_index()
    .sort_values(
        [
            "source_count",
            "asdm_count",
            "spw_count",
        ],
        ascending=False,
    )
)


display(sample_manifest_df)

,member_ous_uid,proposal_id,sample_strata,archive_rows,source_count,spw_count,asdm_count,target_name_count,frequency_min_ghz,frequency_max_ghz,mosaic_state_count,mosaic_state,frequency_support_signature_count,frequency_support_missing_count
0,uid://A001/X11d/Xf,2013.1.00034.S,mid_frequency_non_mosaic,80,20,4,1,20,236.028770,254.031246,1,F,1,0
6,uid://A001/X12d/X6,2013.1.00266.S,low_frequency_non_mosaic,11,1,11,1,1,85.567152,99.399053,1,F,1,0
7,uid://A001/X12d/Xa,2013.1.00266.S,low_frequency_non_mosaic,11,1,11,1,1,85.558407,99.386774,1,F,1,0
15,uid://A001/X62/Xb,2011.0.00780.S,mid_frequency_non_mosaic,8,1,8,1,1,215.076887,235.920481,1,F,1,0
2,uid://A001/X11f/X6,2013.1.00518.S,mid_frequency_mosaic,6,1,6,1,1,217.993209,232.933135,1,T,1,0
3,uid://A001/X11f/X8,2013.1.00518.S,mid_frequency_mosaic,6,1,6,1,1,217.993214,232.933666,1,T,1,0
12,uid://A001/X15bc/Xa,2021.1.00862.S,recent_mid_frequency,5,1,5,1,1,251.864296,267.997572,1,T,1,0
1,uid://A001/X11e/X5,2013.1.00088.S,mid_frequency_non_mosaic,4,1,4,1,1,223.977495,241.975687,1,F,1,0
4,uid://A001/X123/Xc,2013.1.01230.S,high_frequency_non_mosaic,4,1,4,1,1,415.105093,428.782969,1,F,1,0
5,uid://A001/X12a/X9,2013.1.00907.S,high_frequency_non_mosaic,4,1,4,1,1,388.366454,402.100024,1,F,1,0


In [19]:
display(
    sample_manifest_df[
        (sample_manifest_df["source_count"] > 1)
        | (sample_manifest_df["asdm_count"] > 1)
        | (
            sample_manifest_df[
                "frequency_support_signature_count"
            ]
            > 1
        )
        | (
            sample_manifest_df[
                "frequency_support_missing_count"
            ]
            > 0
        )
    ]
)

,member_ous_uid,proposal_id,sample_strata,archive_rows,source_count,spw_count,asdm_count,target_name_count,frequency_min_ghz,frequency_max_ghz,mosaic_state_count,mosaic_state,frequency_support_signature_count,frequency_support_missing_count
0,uid://A001/X11d/Xf,2013.1.00034.S,mid_frequency_non_mosaic,80,20,4,1,20,236.02877,254.031246,1,F,1,0


## Step 4 — Raw Grammar Inventory

Before implementing a parser, the raw strings are inspected for structural
variation.

The experiment records:

- the number of bracketed components;
- the number of `U` separators;
- the number of comma-separated tokens in each component;
- residual text outside the recognized brackets and separators;
- missing and malformed values.

No raw value is discarded when its structure differs from the preliminary
expectation.

In [20]:
BRACKET_COMPONENT_PATTERN = re.compile(
    r"\[([^\[\]]*)\]"
)


def normalize_frequency_support_text(
    value: Any,
) -> str | None:
    if pd.isna(value):
        return None

    if isinstance(value, bytes):
        value = value.decode(
            "utf-8",
            errors="replace",
        )

    normalized_value = str(value).strip()

    if not normalized_value:
        return None

    return normalized_value


def extract_bracket_components(
    value: Any,
) -> list[str]:
    normalized_value = (
        normalize_frequency_support_text(value)
    )

    if normalized_value is None:
        return []

    return [
        component.strip()
        for component in (
            BRACKET_COMPONENT_PATTERN.findall(
                normalized_value
            )
        )
    ]

In [21]:
archive_sample_df[
    "frequency_support_text"
] = archive_sample_df[
    "frequency_support"
].map(
    normalize_frequency_support_text
)


unique_signature_df = (
    archive_sample_df[
        [
            "member_ous_uid",
            "asdm_uid",
            "obs_id_source",
            "frequency_support_text",
        ]
    ]
    .dropna(
        subset=["frequency_support_text"]
    )
    .drop_duplicates(
        subset=["frequency_support_text"]
    )
    .reset_index(drop=True)
)


print(
    "Archive rows:",
    len(archive_sample_df),
)

print(
    "Unique non-missing frequency-support strings:",
    len(unique_signature_df),
)

Archive rows: 171
Unique non-missing frequency-support strings: 18


In [22]:
signature_inventory_records = []

for row in unique_signature_df.itertuples(
    index=False
):
    raw_text = row.frequency_support_text

    components = extract_bracket_components(
        raw_text
    )

    separator_count = len(
        re.findall(
            r"\s+U\s+",
            raw_text,
        )
    )

    residual_text = (
        BRACKET_COMPONENT_PATTERN
        .sub("", raw_text)
    )

    residual_text = re.sub(
        r"\s*U\s*",
        "",
        residual_text,
    ).strip()

    token_counts = [
        len(
            [
                token
                for token in component.split(",")
                if token.strip()
            ]
        )
        for component in components
    ]

    signature_inventory_records.append(
        {
            "member_ous_uid": row.member_ous_uid,
            "asdm_uid": row.asdm_uid,
            "obs_id_source": row.obs_id_source,
            "frequency_support_text": raw_text,
            "component_count": len(components),
            "separator_count": separator_count,
            "expected_separator_count": max(
                len(components) - 1,
                0,
            ),
            "separator_count_matches": (
                separator_count
                == max(len(components) - 1, 0)
            ),
            "component_token_counts": token_counts,
            "residual_text": residual_text,
            "has_unparsed_residual": bool(
                residual_text
            ),
        }
    )


signature_inventory_df = pd.DataFrame(
    signature_inventory_records
)


display(signature_inventory_df.head(20))

,member_ous_uid,asdm_uid,obs_id_source,frequency_support_text,component_count,separator_count,expected_separator_count,separator_count_matches,component_token_counts,residual_text,has_unparsed_residual
0,uid://A001/X6f/Xe,uid://A002/X2c75ca/X7ce,R Scl,"[214.32..216.21GHz,976.56kHz,8.8mJy/beam@10km/s,546.7uJy/beam@native, XX YY] U [216.18..218.07GHz,976.56kHz,9.3mJy/beam@10km/s,575.8uJy/beam@native, XX YY] U [229.32..231.21GHz...",4,3,3,True,"[5, 5, 5, 5]",,False
1,uid://A001/X62/Xb,uid://A002/X331413/X491,HD21997,"[214.84..215.31GHz,244.14kHz,2.4mJy/beam@10km/s,293.7uJy/beam@native, XX YY] U [215.84..216.31GHz,244.14kHz,2.4mJy/beam@10km/s,293.8uJy/beam@native, XX YY] U [219.35..219.82GHz...",8,7,7,True,"[5, 5, 5, 5, 5, 5, 5, 5]",,False
2,uid://A001/X6f/X4,uid://A002/X2c1301/Xa2,SDC335.579-0.292,"[90.62..90.74GHz,61.04kHz,6mJy/beam@10km/s,961.1uJy/beam@native, XX YY] U [93.14..93.25GHz,61.04kHz,5.4mJy/beam@10km/s,876.3uJy/beam@native, XX YY] U [103.99..104.11GHz,61.04kH...",4,3,3,True,"[5, 5, 5, 5]",,False
3,uid://A001/X11f/X8,uid://A002/X822d50/X72c,BHR71_C18O,"[217.03..218.95GHz,31250.00kHz,25.4mJy/beam@10km/s,1.6mJy/beam@native, XX YY] U [219.53..219.59GHz,61.04kHz,25.3mJy/beam@10km/s,8.7mJy/beam@native, XX YY] U [220.37..220.43GHz,...",6,5,5,True,"[5, 5, 5, 5, 5, 5]",,False
4,uid://A001/X11f/X6,uid://A002/X99c183/X25b6,BHR71_C18O,"[217.00..218.99GHz,31250.00kHz,5.4mJy/beam@10km/s,325.9uJy/beam@native, XX YY] U [219.53..219.59GHz,61.04kHz,5.4mJy/beam@10km/s,1.9mJy/beam@native, XX YY] U [220.37..220.43GHz,...",6,5,5,True,"[5, 5, 5, 5, 5, 5]",,False
5,uid://A001/X12a/X9,uid://A002/Xa25bbf/X429f,SPT0346-52,"[387.37..389.36GHz,31250.00kHz,1.6mJy/beam@10km/s,132.2uJy/beam@native, XX YY] U [388.74..390.72GHz,31250.00kHz,1.6mJy/beam@10km/s,132.3uJy/beam@native, XX YY] U [399.31..401.2...",4,3,3,True,"[5, 5, 5, 5]",,False
6,uid://A001/X12d/X6,uid://A002/Xa7dc73/X17cc,g75.78+0.34,"[85.54..85.60GHz,122.07kHz,872.8uJy/beam@10km/s,192.6uJy/beam@native, XX YY] U [85.73..85.79GHz,122.07kHz,871.9uJy/beam@10km/s,192.6uJy/beam@native, XX YY] U [86.21..86.27GHz,6...",11,10,10,True,"[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5]",,False
7,uid://A001/X137/Xd,uid://A002/Xaa96da/X42,Orion_Source_I,"[426.06..427.93GHz,1128.91kHz,8.6mJy/beam@10km/s,745.7uJy/beam@native, XX YY] U [427.83..429.70GHz,1128.91kHz,8.6mJy/beam@10km/s,747uJy/beam@native, XX YY] U [437.92..438.86GHz...",4,3,3,True,"[5, 5, 5, 5]",,False
8,uid://A001/X123/Xc,uid://A002/Xa25bbf/X15f2,G045.1+61.1,"[414.11..416.10GHz,31250.00kHz,10.3mJy/beam@10km/s,854.4uJy/beam@native, XX YY] U [415.81..417.79GHz,31250.00kHz,10.3mJy/beam@10km/s,855.7uJy/beam@native, XX YY] U [425.87..427...",4,3,3,True,"[5, 5, 5, 5]",,False
9,uid://A001/X132/Xa,uid://A002/Xa4fd6e/X3a5,NGC5253,"[99.87..101.86GHz,31250.00kHz,976.8uJy/beam@10km/s,40.1uJy/beam@native, XX YY] U [101.87..103.85GHz,31250.00kHz,967.7uJy/beam@10km/s,40.1uJy/beam@native, XX YY] U [112.86..114....",4,3,3,True,"[5, 5, 5, 5]",,False


In [23]:
component_inventory_records = []

for signature_row in (
    signature_inventory_df.itertuples(
        index=False
    )
):
    components = extract_bracket_components(
        signature_row.frequency_support_text
    )

    for component_index, component_text in enumerate(
        components,
        start=1,
    ):
        tokens = [
            token.strip()
            for token in component_text.split(",")
        ]

        component_inventory_records.append(
            {
                "member_ous_uid": (
                    signature_row.member_ous_uid
                ),
                "asdm_uid": signature_row.asdm_uid,
                "obs_id_source": (
                    signature_row.obs_id_source
                ),
                "component_index": component_index,
                "component_text": component_text,
                "token_count": len(tokens),
                "tokens": tokens,
            }
        )


component_inventory_df = pd.DataFrame(
    component_inventory_records
)


display(component_inventory_df.head(20))

,member_ous_uid,asdm_uid,obs_id_source,component_index,component_text,token_count,tokens
0,uid://A001/X6f/Xe,uid://A002/X2c75ca/X7ce,R Scl,1,"214.32..216.21GHz,976.56kHz,8.8mJy/beam@10km/s,546.7uJy/beam@native, XX YY",5,"[214.32..216.21GHz, 976.56kHz, 8.8mJy/beam@10km/s, 546.7uJy/beam@native, XX YY]"
1,uid://A001/X6f/Xe,uid://A002/X2c75ca/X7ce,R Scl,2,"216.18..218.07GHz,976.56kHz,9.3mJy/beam@10km/s,575.8uJy/beam@native, XX YY",5,"[216.18..218.07GHz, 976.56kHz, 9.3mJy/beam@10km/s, 575.8uJy/beam@native, XX YY]"
2,uid://A001/X6f/Xe,uid://A002/X2c75ca/X7ce,R Scl,3,"229.32..231.21GHz,976.56kHz,9.1mJy/beam@10km/s,579.6uJy/beam@native, XX YY",5,"[229.32..231.21GHz, 976.56kHz, 9.1mJy/beam@10km/s, 579.6uJy/beam@native, XX YY]"
3,uid://A001/X6f/Xe,uid://A002/X2c75ca/X7ce,R Scl,4,"231.18..233.07GHz,976.56kHz,9mJy/beam@10km/s,580.1uJy/beam@native, XX YY",5,"[231.18..233.07GHz, 976.56kHz, 9mJy/beam@10km/s, 580.1uJy/beam@native, XX YY]"
4,uid://A001/X62/Xb,uid://A002/X331413/X491,HD21997,1,"214.84..215.31GHz,244.14kHz,2.4mJy/beam@10km/s,293.7uJy/beam@native, XX YY",5,"[214.84..215.31GHz, 244.14kHz, 2.4mJy/beam@10km/s, 293.7uJy/beam@native, XX YY]"
5,uid://A001/X62/Xb,uid://A002/X331413/X491,HD21997,2,"215.84..216.31GHz,244.14kHz,2.4mJy/beam@10km/s,293.8uJy/beam@native, XX YY",5,"[215.84..216.31GHz, 244.14kHz, 2.4mJy/beam@10km/s, 293.8uJy/beam@native, XX YY]"
6,uid://A001/X62/Xb,uid://A002/X331413/X491,HD21997,3,"219.35..219.82GHz,244.14kHz,2.3mJy/beam@10km/s,291uJy/beam@native, XX YY",5,"[219.35..219.82GHz, 244.14kHz, 2.3mJy/beam@10km/s, 291uJy/beam@native, XX YY]"
7,uid://A001/X62/Xb,uid://A002/X331413/X491,HD21997,4,"219.99..220.46GHz,244.14kHz,2.3mJy/beam@10km/s,291.1uJy/beam@native, XX YY",5,"[219.99..220.46GHz, 244.14kHz, 2.3mJy/beam@10km/s, 291.1uJy/beam@native, XX YY]"
8,uid://A001/X62/Xb,uid://A002/X331413/X491,HD21997,5,"230.49..230.96GHz,244.14kHz,2.3mJy/beam@10km/s,295.9uJy/beam@native, XX YY",5,"[230.49..230.96GHz, 244.14kHz, 2.3mJy/beam@10km/s, 295.9uJy/beam@native, XX YY]"
9,uid://A001/X62/Xb,uid://A002/X331413/X491,HD21997,6,"231.79..232.26GHz,244.14kHz,2.3mJy/beam@10km/s,296.1uJy/beam@native, XX YY",5,"[231.79..232.26GHz, 244.14kHz, 2.3mJy/beam@10km/s, 296.1uJy/beam@native, XX YY]"


In [24]:
print("Component-count distribution:")

display(
    signature_inventory_df[
        "component_count"
    ]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("component_count")
    .reset_index(name="signature_count")
)


print("Token-count distribution:")

display(
    component_inventory_df[
        "token_count"
    ]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("token_count")
    .reset_index(name="component_count")
)


print("Signatures with separator mismatch:")

display(
    signature_inventory_df[
        ~signature_inventory_df[
            "separator_count_matches"
        ]
    ]
)


print("Signatures with unparsed residual text:")

display(
    signature_inventory_df[
        signature_inventory_df[
            "has_unparsed_residual"
        ]
    ]
)

Component-count distribution:


,component_count,signature_count
0,4,12
1,5,1
2,6,2
3,8,1
4,11,2


Token-count distribution:


,token_count,component_count
0,5,95


Signatures with separator mismatch:


,member_ous_uid,asdm_uid,obs_id_source,frequency_support_text,component_count,separator_count,expected_separator_count,separator_count_matches,component_token_counts,residual_text,has_unparsed_residual


Signatures with unparsed residual text:


,member_ous_uid,asdm_uid,obs_id_source,frequency_support_text,component_count,separator_count,expected_separator_count,separator_count_matches,component_token_counts,residual_text,has_unparsed_residual


## Step 5 — Tolerant Preliminary Parser

This experiment converts bracketed frequency-support components into a
long-form table.

The parser is intentionally tolerant:

- the complete raw text is always preserved;
- optional tokens are allowed;
- unknown tokens are retained;
- one malformed component does not stop the complete dataset;
- parser status is recorded as `PARSED`, `PARTIAL`, or `FAILED`.

The resulting schema is preliminary and will be revised after the observed
formats and field-ownership relationships have been analyzed.

In [25]:
NUMBER_PATTERN = (
    r"[+-]?"
    r"(?:\d+(?:\.\d*)?|\.\d+)"
    r"(?:[eE][+-]?\d+)?"
)


FREQUENCY_RANGE_PATTERN = re.compile(
    rf"^\s*"
    rf"(?P<low>{NUMBER_PATTERN})"
    rf"\s*\.\.\s*"
    rf"(?P<high>{NUMBER_PATTERN})"
    rf"\s*(?P<unit>[A-Za-z]+)"
    rf"\s*$"
)


QUANTITY_PATTERN = re.compile(
    rf"^\s*"
    rf"(?P<value>{NUMBER_PATTERN})"
    rf"\s*(?P<unit>.+?)"
    rf"\s*$"
)


SENSITIVITY_PATTERN = re.compile(
    rf"^\s*"
    rf"(?P<value>{NUMBER_PATTERN})"
    rf"\s*(?P<unit>[^@]+?)"
    rf"\s*@\s*"
    rf"(?P<basis>.+?)"
    rf"\s*$"
)

In [26]:
def parse_frequency_support_component(
    component_text: str,
) -> dict[str, Any]:
    tokens = [
        token.strip()
        for token in component_text.split(",")
        if token.strip()
    ]

    parsed: dict[str, Any] = {
        "component_text": component_text,
        "token_count": len(tokens),
        "frequency_low": None,
        "frequency_high": None,
        "frequency_unit": None,
        "resolution_value": None,
        "resolution_unit": None,
        "sensitivity_entries": [],
        "polarization_text": None,
        "unknown_tokens": [],
        "issues": [],
    }

    if not tokens:
        parsed["issues"].append(
            "Component contains no tokens."
        )
        parsed["parse_status"] = "FAILED"
        return parsed

    frequency_match = (
        FREQUENCY_RANGE_PATTERN.match(
            tokens[0]
        )
    )

    if frequency_match is None:
        parsed["issues"].append(
            "Frequency range could not be parsed."
        )
    else:
        parsed["frequency_low"] = float(
            frequency_match.group("low")
        )
        parsed["frequency_high"] = float(
            frequency_match.group("high")
        )
        parsed["frequency_unit"] = (
            frequency_match.group("unit")
        )

    if len(tokens) < 2:
        parsed["issues"].append(
            "Frequency resolution token is missing."
        )
    else:
        resolution_match = QUANTITY_PATTERN.match(
            tokens[1]
        )

        if resolution_match is None:
            parsed["issues"].append(
                "Frequency resolution could not "
                "be parsed."
            )
        else:
            parsed["resolution_value"] = float(
                resolution_match.group("value")
            )
            parsed["resolution_unit"] = (
                resolution_match.group("unit")
                .strip()
            )

    polarization_tokens = []

    for token in tokens[2:]:
        if "@" in token:
            sensitivity_match = (
                SENSITIVITY_PATTERN.match(token)
            )

            if sensitivity_match is None:
                parsed["issues"].append(
                    "Sensitivity token could not "
                    f"be parsed: {token}"
                )
                parsed["unknown_tokens"].append(
                    token
                )
            else:
                parsed[
                    "sensitivity_entries"
                ].append(
                    {
                        "value": float(
                            sensitivity_match.group(
                                "value"
                            )
                        ),
                        "unit": (
                            sensitivity_match.group(
                                "unit"
                            )
                            .strip()
                        ),
                        "basis": (
                            sensitivity_match.group(
                                "basis"
                            )
                            .strip()
                        ),
                        "raw_token": token,
                    }
                )
        else:
            polarization_tokens.append(token)

    if polarization_tokens:
        parsed["polarization_text"] = " ".join(
            polarization_tokens
        )
    else:
        parsed["issues"].append(
            "Polarization token is missing."
        )

    if parsed["frequency_low"] is None:
        parsed["parse_status"] = "FAILED"
    elif parsed["issues"]:
        parsed["parse_status"] = "PARTIAL"
    else:
        parsed["parse_status"] = "PARSED"

    return parsed

In [27]:
parsed_component_records = []

for signature_row in (
    signature_inventory_df.itertuples(
        index=False
    )
):
    raw_text = signature_row.frequency_support_text

    components = extract_bracket_components(
        raw_text
    )

    if not components:
        parsed_component_records.append(
            {
                "member_ous_uid": (
                    signature_row.member_ous_uid
                ),
                "asdm_uid": signature_row.asdm_uid,
                "obs_id_source": (
                    signature_row.obs_id_source
                ),
                "frequency_support_text": raw_text,
                "component_index": None,
                "component_text": None,
                "parse_status": "FAILED",
                "issues": [
                    "No bracketed component found."
                ],
            }
        )

        continue

    for component_index, component_text in enumerate(
        components,
        start=1,
    ):
        parsed_component = (
            parse_frequency_support_component(
                component_text
            )
        )

        parsed_component_records.append(
            {
                "member_ous_uid": (
                    signature_row.member_ous_uid
                ),
                "asdm_uid": signature_row.asdm_uid,
                "obs_id_source": (
                    signature_row.obs_id_source
                ),
                "frequency_support_text": raw_text,
                "component_index": component_index,
                **parsed_component,
            }
        )


parsed_components_df = pd.DataFrame(
    parsed_component_records
)


display(parsed_components_df.head(20))

,member_ous_uid,asdm_uid,obs_id_source,frequency_support_text,component_index,component_text,token_count,frequency_low,frequency_high,frequency_unit,resolution_value,resolution_unit,sensitivity_entries,polarization_text,unknown_tokens,issues,parse_status
0,uid://A001/X6f/Xe,uid://A002/X2c75ca/X7ce,R Scl,"[214.32..216.21GHz,976.56kHz,8.8mJy/beam@10km/s,546.7uJy/beam@native, XX YY] U [216.18..218.07GHz,976.56kHz,9.3mJy/beam@10km/s,575.8uJy/beam@native, XX YY] U [229.32..231.21GHz...",1,"214.32..216.21GHz,976.56kHz,8.8mJy/beam@10km/s,546.7uJy/beam@native, XX YY",5,214.32,216.21,GHz,976.56,kHz,"[{'value': 8.8, 'unit': 'mJy/beam', 'basis': '10km/s', 'raw_token': '8.8mJy/beam@10km/s'}, {'value': 546.7, 'unit': 'uJy/beam', 'basis': 'native', 'raw_token': '546.7uJy/beam@n...",XX YY,[],[],PARSED
1,uid://A001/X6f/Xe,uid://A002/X2c75ca/X7ce,R Scl,"[214.32..216.21GHz,976.56kHz,8.8mJy/beam@10km/s,546.7uJy/beam@native, XX YY] U [216.18..218.07GHz,976.56kHz,9.3mJy/beam@10km/s,575.8uJy/beam@native, XX YY] U [229.32..231.21GHz...",2,"216.18..218.07GHz,976.56kHz,9.3mJy/beam@10km/s,575.8uJy/beam@native, XX YY",5,216.18,218.07,GHz,976.56,kHz,"[{'value': 9.3, 'unit': 'mJy/beam', 'basis': '10km/s', 'raw_token': '9.3mJy/beam@10km/s'}, {'value': 575.8, 'unit': 'uJy/beam', 'basis': 'native', 'raw_token': '575.8uJy/beam@n...",XX YY,[],[],PARSED
2,uid://A001/X6f/Xe,uid://A002/X2c75ca/X7ce,R Scl,"[214.32..216.21GHz,976.56kHz,8.8mJy/beam@10km/s,546.7uJy/beam@native, XX YY] U [216.18..218.07GHz,976.56kHz,9.3mJy/beam@10km/s,575.8uJy/beam@native, XX YY] U [229.32..231.21GHz...",3,"229.32..231.21GHz,976.56kHz,9.1mJy/beam@10km/s,579.6uJy/beam@native, XX YY",5,229.32,231.21,GHz,976.56,kHz,"[{'value': 9.1, 'unit': 'mJy/beam', 'basis': '10km/s', 'raw_token': '9.1mJy/beam@10km/s'}, {'value': 579.6, 'unit': 'uJy/beam', 'basis': 'native', 'raw_token': '579.6uJy/beam@n...",XX YY,[],[],PARSED
3,uid://A001/X6f/Xe,uid://A002/X2c75ca/X7ce,R Scl,"[214.32..216.21GHz,976.56kHz,8.8mJy/beam@10km/s,546.7uJy/beam@native, XX YY] U [216.18..218.07GHz,976.56kHz,9.3mJy/beam@10km/s,575.8uJy/beam@native, XX YY] U [229.32..231.21GHz...",4,"231.18..233.07GHz,976.56kHz,9mJy/beam@10km/s,580.1uJy/beam@native, XX YY",5,231.18,233.07,GHz,976.56,kHz,"[{'value': 9.0, 'unit': 'mJy/beam', 'basis': '10km/s', 'raw_token': '9mJy/beam@10km/s'}, {'value': 580.1, 'unit': 'uJy/beam', 'basis': 'native', 'raw_token': '580.1uJy/beam@nat...",XX YY,[],[],PARSED
4,uid://A001/X62/Xb,uid://A002/X331413/X491,HD21997,"[214.84..215.31GHz,244.14kHz,2.4mJy/beam@10km/s,293.7uJy/beam@native, XX YY] U [215.84..216.31GHz,244.14kHz,2.4mJy/beam@10km/s,293.8uJy/beam@native, XX YY] U [219.35..219.82GHz...",1,"214.84..215.31GHz,244.14kHz,2.4mJy/beam@10km/s,293.7uJy/beam@native, XX YY",5,214.84,215.31,GHz,244.14,kHz,"[{'value': 2.4, 'unit': 'mJy/beam', 'basis': '10km/s', 'raw_token': '2.4mJy/beam@10km/s'}, {'value': 293.7, 'unit': 'uJy/beam', 'basis': 'native', 'raw_token': '293.7uJy/beam@n...",XX YY,[],[],PARSED
5,uid://A001/X62/Xb,uid://A002/X331413/X491,HD21997,"[214.84..215.31GHz,244.14kHz,2.4mJy/beam@10km/s,293.7uJy/beam@native, XX YY] U [215.84..216.31GHz,244.14kHz,2.4mJy/beam@10km/s,293.8uJy/beam@native, XX YY] U [219.35..219.82GHz...",2,"215.84..216.31GHz,244.14kHz,2.4mJy/beam@10km/s,293.8uJy/beam@native, XX YY",5,215.84,216.31,GHz,244.14,kHz,"[{'value': 2.4, 'unit': 'mJy/beam', 'basis': '10km/s', 'raw_token': '2.4mJy/beam@10km/s'}, {'value': 293.8, 'unit': 'uJy/beam', 'basis': 'native', 'raw_token': '293.8uJy/beam@n...",XX YY,[],[],PARSED
6,uid://A001/X62/Xb,uid://A002/X331413/X491,HD21997,"[214.84..215.31GHz,244.14kHz,2.4mJy/beam@10km/s,293.7uJy/beam@native, XX YY] U [215.84..216.31GHz,244.14kHz,2.4mJy/beam@10km/s,293.8uJy/beam@native, XX YY] U [219.35..219.82GHz...",3,"219.35..219.82GHz,244.14kHz,2.3mJy/beam@10km/s,291uJy/beam@native, XX YY",5,219.35,219.82,GHz,244.14,kHz,"[{'value': 2.3, 'unit': 'mJy/beam', 'basis': '10km/s', 'raw_token': '2.3mJy/beam@10km/s'}, {'value': 291.0, 'unit': 'uJy/beam', 'basis': 'nati

In [28]:
def convert_quantity_value(
    value: float | None,
    source_unit: str | None,
    target_unit: u.Unit,
) -> float | None:
    if value is None or source_unit is None:
        return None

    try:
        quantity = value * u.Unit(source_unit)

        return float(
            quantity.to_value(target_unit)
        )
    except Exception:
        return None


parsed_components_df[
    "frequency_low_ghz"
] = parsed_components_df.apply(
    lambda row: convert_quantity_value(
        row.get("frequency_low"),
        row.get("frequency_unit"),
        u.GHz,
    ),
    axis=1,
)


parsed_components_df[
    "frequency_high_ghz"
] = parsed_components_df.apply(
    lambda row: convert_quantity_value(
        row.get("frequency_high"),
        row.get("frequency_unit"),
        u.GHz,
    ),
    axis=1,
)


parsed_components_df[
    "resolution_mhz"
] = parsed_components_df.apply(
    lambda row: convert_quantity_value(
        row.get("resolution_value"),
        row.get("resolution_unit"),
        u.MHz,
    ),
    axis=1,
)

In [29]:
sensitivity_records = []

for parsed_row in parsed_components_df.itertuples(
    index=False
):
    sensitivity_entries = getattr(
        parsed_row,
        "sensitivity_entries",
        None,
    )

    if not isinstance(
        sensitivity_entries,
        list,
    ):
        continue

    for sensitivity_index, entry in enumerate(
        sensitivity_entries,
        start=1,
    ):
        sensitivity_records.append(
            {
                "member_ous_uid": (
                    parsed_row.member_ous_uid
                ),
                "asdm_uid": parsed_row.asdm_uid,
                "obs_id_source": (
                    parsed_row.obs_id_source
                ),
                "component_index": (
                    parsed_row.component_index
                ),
                "sensitivity_index": (
                    sensitivity_index
                ),
                "sensitivity_value": (
                    entry["value"]
                ),
                "sensitivity_unit": entry["unit"],
                "sensitivity_basis": (
                    entry["basis"]
                ),
                "raw_token": entry["raw_token"],
            }
        )


sensitivity_long_df = pd.DataFrame(
    sensitivity_records
)


display(sensitivity_long_df.head(20))

,member_ous_uid,asdm_uid,obs_id_source,component_index,sensitivity_index,sensitivity_value,sensitivity_unit,sensitivity_basis,raw_token
0,uid://A001/X6f/Xe,uid://A002/X2c75ca/X7ce,R Scl,1,1,8.8,mJy/beam,10km/s,8.8mJy/beam@10km/s
1,uid://A001/X6f/Xe,uid://A002/X2c75ca/X7ce,R Scl,1,2,546.7,uJy/beam,native,546.7uJy/beam@native
2,uid://A001/X6f/Xe,uid://A002/X2c75ca/X7ce,R Scl,2,1,9.3,mJy/beam,10km/s,9.3mJy/beam@10km/s
3,uid://A001/X6f/Xe,uid://A002/X2c75ca/X7ce,R Scl,2,2,575.8,uJy/beam,native,575.8uJy/beam@native
4,uid://A001/X6f/Xe,uid://A002/X2c75ca/X7ce,R Scl,3,1,9.1,mJy/beam,10km/s,9.1mJy/beam@10km/s
5,uid://A001/X6f/Xe,uid://A002/X2c75ca/X7ce,R Scl,3,2,579.6,uJy/beam,native,579.6uJy/beam@native
6,uid://A001/X6f/Xe,uid://A002/X2c75ca/X7ce,R Scl,4,1,9.0,mJy/beam,10km/s,9mJy/beam@10km/s
7,uid://A001/X6f/Xe,uid://A002/X2c75ca/X7ce,R Scl,4,2,580.1,uJy/beam,native,580.1uJy/beam@native
8,uid://A001/X62/Xb,uid://A002/X331413/X491,HD21997,1,1,2.4,mJy/beam,10km/s,2.4mJy/beam@10km/s
9,uid://A001/X62/Xb,uid://A002/X331413/X491,HD21997,1,2,293.7,uJy/beam,native,293.7uJy/beam@native


In [30]:
print("Parse-status distribution:")

display(
    parsed_components_df[
        "parse_status"
    ]
    .value_counts(dropna=False)
    .rename_axis("parse_status")
    .reset_index(name="component_count")
)


print("Frequency-unit distribution:")

display(
    parsed_components_df[
        "frequency_unit"
    ]
    .value_counts(dropna=False)
    .rename_axis("frequency_unit")
    .reset_index(name="component_count")
)


print("Resolution-unit distribution:")

display(
    parsed_components_df[
        "resolution_unit"
    ]
    .value_counts(dropna=False)
    .rename_axis("resolution_unit")
    .reset_index(name="component_count")
)


print("Polarization distribution:")

display(
    parsed_components_df[
        "polarization_text"
    ]
    .value_counts(dropna=False)
    .rename_axis("polarization")
    .reset_index(name="component_count")
)


print("Sensitivity-basis distribution:")

if sensitivity_long_df.empty:
    print("No sensitivity entries were parsed.")
else:
    display(
        sensitivity_long_df[
            "sensitivity_basis"
        ]
        .value_counts(dropna=False)
        .rename_axis("sensitivity_basis")
        .reset_index(name="entry_count")
    )

Parse-status distribution:


,parse_status,component_count
0,PARSED,95


Frequency-unit distribution:


,frequency_unit,component_count
0,GHz,95


Resolution-unit distribution:


,resolution_unit,component_count
0,kHz,95


Polarization distribution:


,polarization,component_count
0,XX YY,91
1,XX XY YX YY,4


Sensitivity-basis distribution:


,sensitivity_basis,entry_count
0,10km/s,95
1,native,95


In [31]:
parser_issues_df = (
    parsed_components_df[
        parsed_components_df[
            "parse_status"
        ].isin(
            ["PARTIAL", "FAILED"]
        )
    ]
    [
        [
            "member_ous_uid",
            "asdm_uid",
            "obs_id_source",
            "component_index",
            "component_text",
            "parse_status",
            "issues",
            "unknown_tokens",
        ]
    ]
    .reset_index(drop=True)
)


print(
    "Components requiring review:",
    len(parser_issues_df),
)


display(parser_issues_df.head(50))

Components requiring review: 0


,member_ous_uid,asdm_uid,obs_id_source,component_index,component_text,parse_status,issues,unknown_tokens


## Step 6 — Validate Frequency-Support Components Against Logical SPWs

This experiment tests whether the number of parsed `frequency_support`
components agrees with the number of unique spectral-window identifiers
parsed from `obs_id` for every sampled Member OUS.

Agreement would support the preliminary interpretation that one bracketed
`frequency_support` component corresponds to one logical spectral window.
This remains a sample-based relationship rather than an Archive-wide
constraint.

In [32]:
member_signature_df = (
    archive_sample_df[
        [
            "member_ous_uid",
            "frequency_support_text",
        ]
    ]
    .dropna(subset=["frequency_support_text"])
    .drop_duplicates()
    .reset_index(drop=True)
)


member_component_records = []

for row in member_signature_df.itertuples(index=False):
    components = extract_bracket_components(
        row.frequency_support_text
    )

    for component_index, component_text in enumerate(
        components,
        start=1,
    ):
        parsed = parse_frequency_support_component(
            component_text
        )

        member_component_records.append(
            {
                "member_ous_uid": row.member_ous_uid,
                "frequency_support_text": (
                    row.frequency_support_text
                ),
                "component_index": component_index,
                **parsed,
            }
        )


member_parsed_components_df = pd.DataFrame(
    member_component_records
)


member_parsed_components_df[
    "frequency_low_ghz"
] = member_parsed_components_df.apply(
    lambda row: convert_quantity_value(
        row["frequency_low"],
        row["frequency_unit"],
        u.GHz,
    ),
    axis=1,
)


member_parsed_components_df[
    "frequency_high_ghz"
] = member_parsed_components_df.apply(
    lambda row: convert_quantity_value(
        row["frequency_high"],
        row["frequency_unit"],
        u.GHz,
    ),
    axis=1,
)


member_parsed_components_df[
    "resolution_mhz"
] = member_parsed_components_df.apply(
    lambda row: convert_quantity_value(
        row["resolution_value"],
        row["resolution_unit"],
        u.MHz,
    ),
    axis=1,
)


member_parsed_components_df[
    "parsed_bandwidth_ghz"
] = (
    member_parsed_components_df["frequency_high_ghz"]
    - member_parsed_components_df["frequency_low_ghz"]
)


member_parsed_components_df[
    "parsed_center_ghz"
] = (
    member_parsed_components_df["frequency_high_ghz"]
    + member_parsed_components_df["frequency_low_ghz"]
) / 2


display(member_parsed_components_df.head())

,member_ous_uid,frequency_support_text,component_index,component_text,token_count,frequency_low,frequency_high,frequency_unit,resolution_value,resolution_unit,sensitivity_entries,polarization_text,unknown_tokens,issues,parse_status,frequency_low_ghz,frequency_high_ghz,resolution_mhz,parsed_bandwidth_ghz,parsed_center_ghz
0,uid://A001/X6f/Xe,"[214.32..216.21GHz,976.56kHz,8.8mJy/beam@10km/s,546.7uJy/beam@native, XX YY] U [216.18..218.07GHz,976.56kHz,9.3mJy/beam@10km/s,575.8uJy/beam@native, XX YY] U [229.32..231.21GHz...",1,"214.32..216.21GHz,976.56kHz,8.8mJy/beam@10km/s,546.7uJy/beam@native, XX YY",5,214.32,216.21,GHz,976.56,kHz,"[{'value': 8.8, 'unit': 'mJy/beam', 'basis': '10km/s', 'raw_token': '8.8mJy/beam@10km/s'}, {'value': 546.7, 'unit': 'uJy/beam', 'basis': 'native', 'raw_token': '546.7uJy/beam@n...",XX YY,[],[],PARSED,214.32,216.21,0.97656,1.89,215.265
1,uid://A001/X6f/Xe,"[214.32..216.21GHz,976.56kHz,8.8mJy/beam@10km/s,546.7uJy/beam@native, XX YY] U [216.18..218.07GHz,976.56kHz,9.3mJy/beam@10km/s,575.8uJy/beam@native, XX YY] U [229.32..231.21GHz...",2,"216.18..218.07GHz,976.56kHz,9.3mJy/beam@10km/s,575.8uJy/beam@native, XX YY",5,216.18,218.07,GHz,976.56,kHz,"[{'value': 9.3, 'unit': 'mJy/beam', 'basis': '10km/s', 'raw_token': '9.3mJy/beam@10km/s'}, {'value': 575.8, 'unit': 'uJy/beam', 'basis': 'native', 'raw_token': '575.8uJy/beam@n...",XX YY,[],[],PARSED,216.18,218.07,0.97656,1.89,217.125
2,uid://A001/X6f/Xe,"[214.32..216.21GHz,976.56kHz,8.8mJy/beam@10km/s,546.7uJy/beam@native, XX YY] U [216.18..218.07GHz,976.56kHz,9.3mJy/beam@10km/s,575.8uJy/beam@native, XX YY] U [229.32..231.21GHz...",3,"229.32..231.21GHz,976.56kHz,9.1mJy/beam@10km/s,579.6uJy/beam@native, XX YY",5,229.32,231.21,GHz,976.56,kHz,"[{'value': 9.1, 'unit': 'mJy/beam', 'basis': '10km/s', 'raw_token': '9.1mJy/beam@10km/s'}, {'value': 579.6, 'unit': 'uJy/beam', 'basis': 'native', 'raw_token': '579.6uJy/beam@n...",XX YY,[],[],PARSED,229.32,231.21,0.97656,1.89,230.265
3,uid://A001/X6f/Xe,"[214.32..216.21GHz,976.56kHz,8.8mJy/beam@10km/s,546.7uJy/beam@native, XX YY] U [216.18..218.07GHz,976.56kHz,9.3mJy/beam@10km/s,575.8uJy/beam@native, XX YY] U [229.32..231.21GHz...",4,"231.18..233.07GHz,976.56kHz,9mJy/beam@10km/s,580.1uJy/beam@native, XX YY",5,231.18,233.07,GHz,976.56,kHz,"[{'value': 9.0, 'unit': 'mJy/beam', 'basis': '10km/s', 'raw_token': '9mJy/beam@10km/s'}, {'value': 580.1, 'unit': 'uJy/beam', 'basis': 'native', 'raw_token': '580.1uJy/beam@nat...",XX YY,[],[],PARSED,231.18,233.07,0.97656,1.89,232.125
4,uid://A001/X62/Xb,"[214.84..215.31GHz,244.14kHz,2.4mJy/beam@10km/s,293.7uJy/beam@native, XX YY] U [215.84..216.31GHz,244.14kHz,2.4mJy/beam@10km/s,293.8uJy/beam@native, XX YY] U [219.35..219.82GHz...",1,"214.84..215.31GHz,244.14kHz,2.4mJy/beam@10km/s,293.7uJy/beam@native, XX YY",5,214.84,215.31,GHz,244.14,kHz,"[{'value': 2.4, 'unit': 'mJy/beam', 'basis': '10km/s', 'raw_token': '2.4mJy/beam@10km/s'}, {'value': 293.7, 'unit': 'uJy/beam', 'basis': 'native', 'raw_token': '293.7uJy/beam@n...",XX YY,[],[],PARSED,214.84,215.31,0.24414,0.47,215.075


In [33]:
component_count_df = (
    member_parsed_components_df
    .groupby("member_ous_uid")
    .agg(
        parsed_component_count=(
            "component_index",
            "count",
        ),
        signature_count=(
            "frequency_support_text",
            "nunique",
        ),
    )
    .reset_index()
)


spw_component_validation_df = (
    sample_manifest_df[
        [
            "member_ous_uid",
            "archive_rows",
            "source_count",
            "spw_count",
            "asdm_count",
        ]
    ]
    .merge(
        component_count_df,
        on="member_ous_uid",
        how="left",
        validate="one_to_one",
    )
)


spw_component_validation_df[
    "component_count_matches_spw_count"
] = (
    spw_component_validation_df[
        "parsed_component_count"
    ]
    == spw_component_validation_df["spw_count"]
)


spw_component_validation_df[
    "source_spw_grid_expected_rows"
] = (
    spw_component_validation_df["source_count"]
    * spw_component_validation_df["spw_count"]
)


spw_component_validation_df[
    "source_spw_grid_matches_rows"
] = (
    spw_component_validation_df[
        "source_spw_grid_expected_rows"
    ]
    == spw_component_validation_df["archive_rows"]
)


display(spw_component_validation_df)


print(
    "Members with matching component and SPW counts:",
    spw_component_validation_df[
        "component_count_matches_spw_count"
    ].sum(),
    "/",
    len(spw_component_validation_df),
)


print(
    "Members forming complete source × SPW grids:",
    spw_component_validation_df[
        "source_spw_grid_matches_rows"
    ].sum(),
    "/",
    len(spw_component_validation_df),
)

,member_ous_uid,archive_rows,source_count,spw_count,asdm_count,parsed_component_count,signature_count,component_count_matches_spw_count,source_spw_grid_expected_rows,source_spw_grid_matches_rows
0,uid://A001/X11d/Xf,80,20,4,1,4,1,True,80,True
1,uid://A001/X12d/X6,11,1,11,1,11,1,True,11,True
2,uid://A001/X12d/Xa,11,1,11,1,11,1,True,11,True
3,uid://A001/X62/Xb,8,1,8,1,8,1,True,8,True
4,uid://A001/X11f/X6,6,1,6,1,6,1,True,6,True
5,uid://A001/X11f/X8,6,1,6,1,6,1,True,6,True
6,uid://A001/X15bc/Xa,5,1,5,1,5,1,True,5,True
7,uid://A001/X11e/X5,4,1,4,1,4,1,True,4,True
8,uid://A001/X123/Xc,4,1,4,1,4,1,True,4,True
9,uid://A001/X12a/X9,4,1,4,1,4,1,True,4,True


Members with matching component and SPW counts: 18 / 18
Members forming complete source × SPW grids: 18 / 18


## Step 7 — Match Archive-Row SPWs to Parsed Frequency Intervals

Equal counts alone do not prove that a parsed component represents the same
spectral window as an Archive row. This experiment reconstructs one row-level
record per Member OUS and SPW, and assigns those records one-to-one to parsed
frequency-support intervals.

The assignment minimizes the combined absolute difference in central
frequency and bandwidth. The resulting differences are diagnostic measurements,
not scientific equivalence thresholds.

In [34]:
logical_spw_df = (
    archive_sample_df
    .groupby(
        [
            "member_ous_uid",
            "spw_identifier",
        ],
        dropna=False,
    )
    .agg(
        archive_row_count=("obs_id", "size"),
        source_count=("obs_id_source", "nunique"),
        row_frequency_median_ghz=("frequency", "median"),
        row_frequency_min_ghz=("frequency", "min"),
        row_frequency_max_ghz=("frequency", "max"),
        row_bandwidth_median_hz=("bandwidth", "median"),
        row_bandwidth_min_hz=("bandwidth", "min"),
        row_bandwidth_max_hz=("bandwidth", "max"),
    )
    .reset_index()
)


logical_spw_df[
    "row_bandwidth_median_ghz"
] = (
    logical_spw_df["row_bandwidth_median_hz"]
    / 1e9
)


logical_spw_df[
    "row_frequency_spread_mhz"
] = (
    logical_spw_df["row_frequency_max_ghz"]
    - logical_spw_df["row_frequency_min_ghz"]
) * 1000


logical_spw_df[
    "row_bandwidth_spread_mhz"
] = (
    logical_spw_df["row_bandwidth_max_hz"]
    - logical_spw_df["row_bandwidth_min_hz"]
) / 1e6


display(logical_spw_df.head(20))

,member_ous_uid,spw_identifier,archive_row_count,source_count,row_frequency_median_ghz,row_frequency_min_ghz,row_frequency_max_ghz,row_bandwidth_median_hz,row_bandwidth_min_hz,row_bandwidth_max_hz,row_bandwidth_median_ghz,row_frequency_spread_mhz,row_bandwidth_spread_mhz
0,uid://A001/X11d/Xf,11,20,20,238.029201,238.029014,238.029278,2.000000e+09,2.000000e+09,2.000000e+09,2.000000,0.263389,0.0
1,uid://A001/X11d/Xf,19,20,20,252.030919,252.030721,252.031000,2.000000e+09,2.000000e+09,2.000000e+09,2.000000,0.278905,0.0
2,uid://A001/X11d/Xf,21,20,20,254.031164,254.030965,254.031246,2.000000e+09,2.000000e+09,2.000000e+09,2.000000,0.281122,0.0
3,uid://A001/X11d/Xf,9,20,20,236.028956,236.028770,236.029032,2.000000e+09,2.000000e+09,2.000000e+09,2.000000,0.261172,0.0
4,uid://A001/X11e/X5,11,1,1,225.977294,225.977294,225.977294,2.000000e+09,2.000000e+09,2.000000e+09,2.000000,0.000000,0.0
5,uid://A001/X11e/X5,19,1,1,239.975888,239.975888,239.975888,2.000000e+09,2.000000e+09,2.000000e+09,2.000000,0.000000,0.0
6,uid://A001/X11e/X5,21,1,1,241.975687,241.975687,241.975687,2.000000e+09,2.000000e+09,2.000000e+09,2.000000,0.000000,0.0
7,uid://A001/X11e/X5,9,1,1,223.977495,223.977495,223.977495,2.000000e+09,2.000000e+09,2.000000e+09,2.000000,0.000000,0.0
8,uid://A001/X11f/X6,17,1,1,219.558709,219.558709,219.558709,5.859375e+07,5.859375e+07,5.859375e+07,0.058594,0.000000,0.0
9,uid://A001/X11f/X6,19,1,1,220.397037,220.397037,220.397037,5.859375e+07,5.859375e+07,5.859375e+07,0.058594,0.000000,0.0


In [35]:
from functools import lru_cache


def assign_components_to_spws(
    spw_group: pd.DataFrame,
    component_group: pd.DataFrame,
) -> tuple[float, list[tuple[int, int]]]:
    spws = spw_group.reset_index(drop=True)
    components = component_group.reset_index(drop=True)

    if len(spws) != len(components):
        raise ValueError(
            "SPW and component counts must agree "
            "before one-to-one assignment."
        )

    number_of_items = len(spws)

    cost_matrix = np.zeros(
        (number_of_items, number_of_items),
        dtype=float,
    )

    for spw_index, spw_row in spws.iterrows():
        for component_index, component_row in (
            components.iterrows()
        ):
            center_difference_mhz = abs(
                spw_row["row_frequency_median_ghz"]
                - component_row["parsed_center_ghz"]
            ) * 1000

            bandwidth_difference_mhz = abs(
                spw_row["row_bandwidth_median_ghz"]
                - component_row["parsed_bandwidth_ghz"]
            ) * 1000

            cost_matrix[
                spw_index,
                component_index,
            ] = (
                center_difference_mhz
                + bandwidth_difference_mhz
            )

    @lru_cache(maxsize=None)
    def solve(
        spw_index: int,
        used_component_mask: int,
    ):
        if spw_index == number_of_items:
            return 0.0, ()

        best_cost = float("inf")
        best_pairs = ()

        for component_index in range(number_of_items):
            component_bit = 1 << component_index

            if used_component_mask & component_bit:
                continue

            remaining_cost, remaining_pairs = solve(
                spw_index + 1,
                used_component_mask | component_bit,
            )

            total_cost = (
                cost_matrix[
                    spw_index,
                    component_index,
                ]
                + remaining_cost
            )

            if total_cost < best_cost:
                best_cost = total_cost
                best_pairs = (
                    (spw_index, component_index),
                    *remaining_pairs,
                )

        return best_cost, best_pairs

    total_cost, assignment_pairs = solve(0, 0)

    return total_cost, list(assignment_pairs)

In [36]:
assignment_records = []
assignment_skipped_records = []


for member_uid, spw_group in logical_spw_df.groupby(
    "member_ous_uid"
):
    component_group = member_parsed_components_df[
        member_parsed_components_df["member_ous_uid"]
        == member_uid
    ].copy()

    if len(spw_group) != len(component_group):
        assignment_skipped_records.append(
            {
                "member_ous_uid": member_uid,
                "spw_count": len(spw_group),
                "component_count": len(component_group),
                "reason": "Count mismatch",
            }
        )
        continue

    total_cost, pairs = assign_components_to_spws(
        spw_group,
        component_group,
    )

    spws = spw_group.reset_index(drop=True)
    components = component_group.reset_index(drop=True)

    for spw_index, component_index in pairs:
        spw_row = spws.iloc[spw_index]
        component_row = components.iloc[component_index]

        assignment_records.append(
            {
                "member_ous_uid": member_uid,
                "spw_identifier": (
                    spw_row["spw_identifier"]
                ),
                "component_index": (
                    component_row["component_index"]
                ),
                "row_center_ghz": (
                    spw_row[
                        "row_frequency_median_ghz"
                    ]
                ),
                "parsed_low_ghz": (
                    component_row[
                        "frequency_low_ghz"
                    ]
                ),
                "parsed_high_ghz": (
                    component_row[
                        "frequency_high_ghz"
                    ]
                ),
                "parsed_center_ghz": (
                    component_row[
                        "parsed_center_ghz"
                    ]
                ),
                "row_bandwidth_ghz": (
                    spw_row[
                        "row_bandwidth_median_ghz"
                    ]
                ),
                "parsed_bandwidth_ghz": (
                    component_row[
                        "parsed_bandwidth_ghz"
                    ]
                ),
                "member_total_assignment_cost": (
                    total_cost
                ),
            }
        )


spw_interval_assignment_df = pd.DataFrame(
    assignment_records
)

assignment_skipped_df = pd.DataFrame(
    assignment_skipped_records
)


print("Matched SPWs:", len(spw_interval_assignment_df))
print("Skipped Member OUS datasets:", len(assignment_skipped_df))

display(assignment_skipped_df)
display(spw_interval_assignment_df.head(20))

Matched SPWs: 95
Skipped Member OUS datasets: 0


""


,member_ous_uid,spw_identifier,component_index,row_center_ghz,parsed_low_ghz,parsed_high_ghz,parsed_center_ghz,row_bandwidth_ghz,parsed_bandwidth_ghz,member_total_assignment_cost
0,uid://A001/X11d/Xf,11,2,238.029201,237.04,239.02,238.030,2.000000,1.98,83.925773
1,uid://A001/X11d/Xf,19,3,252.030919,251.04,253.02,252.030,2.000000,1.98,83.925773
2,uid://A001/X11d/Xf,21,4,254.031164,253.04,255.02,254.030,2.000000,1.98,83.925773
3,uid://A001/X11d/Xf,9,1,236.028956,235.04,237.02,236.030,2.000000,1.98,83.925773
4,uid://A001/X11e/X5,11,2,225.977294,224.99,226.97,225.980,2.000000,1.98,66.784968
5,uid://A001/X11e/X5,19,3,239.975888,238.98,240.97,239.975,2.000000,1.99,66.784968
6,uid://A001/X11e/X5,21,4,241.975687,240.98,242.97,241.975,2.000000,1.99,66.784968
7,uid://A001/X11e/X5,9,1,223.977495,222.99,224.97,223.980,2.000000,1.98,66.784968
8,uid://A001/X11f/X6,17,2,219.558709,219.53,219.59,219.560,0.058594,0.06,39.709860
9,uid://A001/X11f/X6,19,3,220.397037,220.37,220.43,220.400,0.058594,0.06,39.709860


## Step 8 — Quantify Row-to-Interval Numerical Differences

This experiment measures how closely the Archive row-level `frequency` and
`bandwidth` values agree with the centre and width calculated from the assigned
`frequency_support` interval.

No scientific matching threshold is defined here. The purpose is to describe
the numerical relationship and identify records requiring manual inspection.

In [37]:
spw_interval_assignment_df[
    "center_difference_mhz"
] = (
    spw_interval_assignment_df["row_center_ghz"]
    - spw_interval_assignment_df[
        "parsed_center_ghz"
    ]
) * 1000


spw_interval_assignment_df[
    "absolute_center_difference_mhz"
] = spw_interval_assignment_df[
    "center_difference_mhz"
].abs()


spw_interval_assignment_df[
    "bandwidth_difference_mhz"
] = (
    spw_interval_assignment_df["row_bandwidth_ghz"]
    - spw_interval_assignment_df[
        "parsed_bandwidth_ghz"
    ]
) * 1000


spw_interval_assignment_df[
    "absolute_bandwidth_difference_mhz"
] = spw_interval_assignment_df[
    "bandwidth_difference_mhz"
].abs()


spw_interval_assignment_df[
    "row_center_inside_parsed_interval"
] = (
    (
        spw_interval_assignment_df["row_center_ghz"]
        >= spw_interval_assignment_df[
            "parsed_low_ghz"
        ]
    )
    & (
        spw_interval_assignment_df["row_center_ghz"]
        <= spw_interval_assignment_df[
            "parsed_high_ghz"
        ]
    )
)


print("Numerical difference summary:")

display(
    spw_interval_assignment_df[
        [
            "absolute_center_difference_mhz",
            "absolute_bandwidth_difference_mhz",
        ]
    ].describe(
        percentiles=[
            0.5,
            0.9,
            0.95,
            0.99,
        ]
    )
)


print("Row centres inside assigned intervals:")

display(
    spw_interval_assignment_df[
        "row_center_inside_parsed_interval"
    ]
    .value_counts(dropna=False)
    .rename_axis("inside_interval")
    .reset_index(name="spw_count")
)


largest_differences_df = (
    spw_interval_assignment_df
    .sort_values(
        [
            "absolute_bandwidth_difference_mhz",
            "absolute_center_difference_mhz",
        ],
        ascending=False,
    )
    .head(20)
)


display(
    largest_differences_df[
        [
            "member_ous_uid",
            "spw_identifier",
            "component_index",
            "row_center_ghz",
            "parsed_low_ghz",
            "parsed_high_ghz",
            "center_difference_mhz",
            "row_bandwidth_ghz",
            "parsed_bandwidth_ghz",
            "bandwidth_difference_mhz",
            "row_center_inside_parsed_interval",
        ]
    ]
)

Numerical difference summary:


,absolute_center_difference_mhz,absolute_bandwidth_difference_mhz
count,95.000000,95.000000
mean,1.643725,8.707237
std,1.221632,8.245086
min,0.024000,1.250000
50%,1.379085,5.000000
90%,3.620456,20.000000
95%,4.026779,20.000000
99%,4.415889,30.000000
max,4.705302,30.000000


Row centres inside assigned intervals:


,inside_interval,spw_count
0,True,95


,member_ous_uid,spw_identifier,component_index,row_center_ghz,parsed_low_ghz,parsed_high_ghz,center_difference_mhz,row_bandwidth_ghz,parsed_bandwidth_ghz,bandwidth_difference_mhz,row_center_inside_parsed_interval
78,uid://A001/X2d1f/X5,30,4,241.982461,241.00,242.97,-2.538757,2.0,1.97,30.0,True
77,uid://A001/X2d1f/X5,26,3,239.982606,239.00,240.97,-2.393798,2.0,1.97,30.0,True
76,uid://A001/X2d1f/X5,22,2,225.983621,225.00,226.97,-1.379085,2.0,1.97,30.0,True
75,uid://A001/X2d1f/X5,18,1,223.983766,223.00,224.97,-1.234126,2.0,1.97,30.0,True
24,uid://A001/X12a/X9,17,4,402.100024,401.11,403.09,0.024108,2.0,1.98,20.0,True
4,uid://A001/X11e/X5,11,2,225.977294,224.99,226.97,-2.705836,2.0,1.98,20.0,True
7,uid://A001/X11e/X5,9,1,223.977495,222.99,224.97,-2.504896,2.0,1.98,20.0,True
53,uid://A001/X132/Xa,23,2,102.861385,101.87,103.85,1.384877,2.0,1.98,20.0,True
51,uid://A001/X132/Xa,19,3,113.849342,112.86,114.84,-0.658027,2.0,1.98,20.0,True
66,uid://A001/X15bc/Xa,23,5,267.997572,267.01,268.99,-2.428124,2.0,1.98,20.0,True


## Step 9 — Test Frequency-Support Ownership in a Multi-ASDM Dataset

The initial frequency-support sample contained only one ASDM UID per Member
OUS. It therefore could not determine whether `frequency_support` belongs to
the Member OUS, ASDM execution, source context, or individual row.

This targeted experiment revisits a multi-ASDM Member OUS identified in
Notebook 2 and compares frequency-support signatures across executions and
sources.

In [38]:
multi_asdm_member_uid = "uid://A001/X3833/X1022"


multi_asdm_count_query = f"""
SELECT COUNT(*) AS total_rows
FROM ivoa.obscore
WHERE science_observation = 'T'
AND member_ous_uid = {
    quote_adql_string(multi_asdm_member_uid)
}
"""


multi_asdm_count_table = run_tap_query(
    multi_asdm_count_query,
    maxrec=10,
    label="Count targeted multi-ASDM rows",
)


multi_asdm_total_rows = int(
    multi_asdm_count_table["total_rows"][0]
)


print(
    "Expected rows for targeted Member OUS:",
    multi_asdm_total_rows,
)


if multi_asdm_total_rows > 5000:
    raise RuntimeError(
        "The targeted Member OUS is unexpectedly large. "
        "Inspect the count before downloading it."
    )

Running: Count targeted multi-ASDM rows
Retrieved 1 rows in 5.2 seconds.
Expected rows for targeted Member OUS: 8


In [39]:
multi_asdm_query = f"""
SELECT
    {column_sql}
FROM ivoa.obscore
WHERE science_observation = 'T'
AND member_ous_uid = {
    quote_adql_string(multi_asdm_member_uid)
}
"""


multi_asdm_table = run_tap_query(
    multi_asdm_query,
    maxrec=max(multi_asdm_total_rows, 1),
    label="Retrieve targeted multi-ASDM dataset",
)


multi_asdm_df = multi_asdm_table.to_pandas().copy()


multi_asdm_parsed_obs_ids = (
    multi_asdm_df["obs_id"].map(parse_obs_id)
)


multi_asdm_df["obs_id_source"] = [
    value[0]
    for value in multi_asdm_parsed_obs_ids
]


multi_asdm_df["spw_identifier"] = [
    value[1]
    for value in multi_asdm_parsed_obs_ids
]


multi_asdm_df[
    "frequency_support_text"
] = multi_asdm_df[
    "frequency_support"
].map(
    normalize_frequency_support_text
)


print("Retrieved rows:", len(multi_asdm_df))
print("ASDM count:", multi_asdm_df["asdm_uid"].nunique())
print(
    "Source-context count:",
    multi_asdm_df["obs_id_source"].nunique(),
)
print(
    "SPW count:",
    multi_asdm_df["spw_identifier"].nunique(),
)
print(
    "Exact frequency-support signatures:",
    multi_asdm_df[
        "frequency_support_text"
    ].nunique(),
)

Running: Retrieve targeted multi-ASDM dataset
Retrieved 8 rows in 1.3 seconds.
Retrieved rows: 8
ASDM count: 2
Source-context count: 2
SPW count: 4
Exact frequency-support signatures: 2


In [40]:
multi_asdm_context_summary_df = (
    multi_asdm_df
    .groupby(
        [
            "asdm_uid",
            "obs_id_source",
        ],
        dropna=False,
    )
    .agg(
        archive_rows=("obs_id", "size"),
        spw_count=("spw_identifier", "nunique"),
        signature_count=(
            "frequency_support_text",
            "nunique",
        ),
        frequency_min_ghz=("frequency", "min"),
        frequency_max_ghz=("frequency", "max"),
        t_min=("t_min", "min"),
        t_max=("t_max", "max"),
        antenna_array_count=(
            "antenna_arrays",
            "nunique",
        ),
    )
    .reset_index()
)


display(multi_asdm_context_summary_df)


signature_assignment_df = (
    multi_asdm_df[
        [
            "member_ous_uid",
            "asdm_uid",
            "obs_id_source",
            "frequency_support_text",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "asdm_uid",
            "obs_id_source",
        ]
    )
    .reset_index(drop=True)
)


display(signature_assignment_df)

,asdm_uid,obs_id_source,archive_rows,spw_count,signature_count,frequency_min_ghz,frequency_max_ghz,t_min,t_max,antenna_array_count
0,uid://A002/X139bfe4/X11e15,SPT2349-56_Core,4,4,1,345.382156,359.309397,61161.618347,61247.324103,1
1,uid://A002/X139fbe0/X7472,SPT2349-56_N1-N2,4,4,1,345.382200,359.309447,61164.618683,61247.350143,1


,member_ous_uid,asdm_uid,obs_id_source,frequency_support_text
0,uid://A001/X3833/X1022,uid://A002/X139bfe4/X11e15,SPT2349-56_Core,"[344.45..346.32GHz,7812.01kHz,24.6mJy/beam@10km/s,1.9mJy/beam@native, XX YY] U [346.32..348.19GHz,7812.01kHz,24.5mJy/beam@10km/s,1.9mJy/beam@native, XX YY] U [356.68..358.55GHz..."
1,uid://A001/X3833/X1022,uid://A002/X139fbe0/X7472,SPT2349-56_N1-N2,"[344.45..346.32GHz,7812.01kHz,21.8mJy/beam@10km/s,1.7mJy/beam@native, XX YY] U [346.32..348.19GHz,7812.01kHz,21.7mJy/beam@10km/s,1.7mJy/beam@native, XX YY] U [356.68..358.55GHz..."


## Step 10 — Search for Format Exceptions and Define a Parser Contract

The first sample contained only complete and regular frequency-support
strings. A production parser must also handle missing fields, alternative
units, unexpected polarization products, malformed intervals, and additional
tokens without silently discarding information.

This step combines a targeted Archive search with controlled boundary cases.
Synthetic cases are parser tests rather than evidence about Archive
frequencies.

In [41]:
format_exception_query = """
SELECT TOP 100
    proposal_id,
    member_ous_uid,
    asdm_uid,
    obs_id,
    target_name,
    frequency_support
FROM ivoa.obscore
WHERE science_observation = 'T'
AND frequency_support IS NOT NULL
AND (
    frequency_support NOT LIKE '%GHz%'
    OR frequency_support NOT LIKE '%kHz%'
    OR frequency_support NOT LIKE '%@10km/s%'
    OR frequency_support NOT LIKE '%@native%'
)
"""


format_exception_table = run_tap_query(
    format_exception_query,
    maxrec=100,
    label="Search for frequency-support format exceptions",
)


format_exception_df = (
    format_exception_table
    .to_pandas()
    .copy()
)


print(
    "Potential format-exception rows:",
    len(format_exception_df),
)


display(format_exception_df.head(30))

Running: Search for frequency-support format exceptions
Retrieved 0 rows in 3.1 seconds.
Potential format-exception rows: 0


,proposal_id,member_ous_uid,asdm_uid,obs_id,target_name,frequency_support


In [42]:
format_exception_parse_records = []


for row in format_exception_df.itertuples(index=False):
    raw_text = normalize_frequency_support_text(
        row.frequency_support
    )

    components = extract_bracket_components(raw_text)

    if not components:
        format_exception_parse_records.append(
            {
                "member_ous_uid": row.member_ous_uid,
                "obs_id": row.obs_id,
                "component_index": None,
                "component_text": None,
                "parse_status": "FAILED",
                "issues": [
                    "No bracketed component found."
                ],
            }
        )
        continue

    for component_index, component_text in enumerate(
        components,
        start=1,
    ):
        parsed = parse_frequency_support_component(
            component_text
        )

        format_exception_parse_records.append(
            {
                "member_ous_uid": row.member_ous_uid,
                "obs_id": row.obs_id,
                "component_index": component_index,
                **parsed,
            }
        )


format_exception_parse_df = pd.DataFrame(
    format_exception_parse_records
)


if format_exception_parse_df.empty:
    print("No format-exception components were returned.")
else:
    display(
        format_exception_parse_df[
            "parse_status"
        ]
        .value_counts(dropna=False)
        .rename_axis("parse_status")
        .reset_index(name="component_count")
    )

    display(
        format_exception_parse_df[
            format_exception_parse_df[
                "parse_status"
            ] != "PARSED"
        ].head(50)
    )

No format-exception components were returned.


In [43]:
parser_boundary_cases = [
    {
        "case": "standard",
        "component": (
            "214.32..216.21GHz,"
            "976.56kHz,"
            "1.0mJy/beam@10km/s,"
            "0.2mJy/beam@native,"
            "XX YY"
        ),
    },
    {
        "case": "full_polarization",
        "component": (
            "214.32..216.21GHz,"
            "976.56kHz,"
            "1.0mJy/beam@10km/s,"
            "0.2mJy/beam@native,"
            "XX XY YX YY"
        ),
    },
    {
        "case": "resolution_in_MHz",
        "component": (
            "214.32..216.21GHz,"
            "0.97656MHz,"
            "1.0mJy/beam@10km/s,"
            "0.2mJy/beam@native,"
            "XX YY"
        ),
    },
    {
        "case": "sensitivity_in_uJy",
        "component": (
            "214.32..216.21GHz,"
            "976.56kHz,"
            "872.8uJy/beam@10km/s,"
            "200uJy/beam@native,"
            "XX YY"
        ),
    },
    {
        "case": "reversed_frequency_range",
        "component": (
            "216.21..214.32GHz,"
            "976.56kHz,"
            "1.0mJy/beam@10km/s,"
            "0.2mJy/beam@native,"
            "XX YY"
        ),
    },
    {
        "case": "missing_native_sensitivity",
        "component": (
            "214.32..216.21GHz,"
            "976.56kHz,"
            "1.0mJy/beam@10km/s,"
            "XX YY"
        ),
    },
    {
        "case": "unknown_polarization_token",
        "component": (
            "214.32..216.21GHz,"
            "976.56kHz,"
            "1.0mJy/beam@10km/s,"
            "0.2mJy/beam@native,"
            "XX YY UNKNOWN"
        ),
    },
    {
        "case": "invalid_frequency_unit",
        "component": (
            "214.32..216.21Banana,"
            "976.56kHz,"
            "1.0mJy/beam@10km/s,"
            "0.2mJy/beam@native,"
            "XX YY"
        ),
    },
    {
        "case": "malformed_frequency_range",
        "component": (
            "214.32-216.21GHz,"
            "976.56kHz,"
            "1.0mJy/beam@10km/s,"
            "0.2mJy/beam@native,"
            "XX YY"
        ),
    },
]

In [44]:
VALID_POLARIZATION_PRODUCTS = {
    "XX",
    "YY",
    "XY",
    "YX",
}


def validate_parsed_component(
    parsed: dict[str, Any],
) -> list[str]:
    validation_issues = list(
        parsed.get("issues", [])
    )

    low = parsed.get("frequency_low")
    high = parsed.get("frequency_high")
    frequency_unit = parsed.get("frequency_unit")

    if (
        low is not None
        and high is not None
        and low >= high
    ):
        validation_issues.append(
            "Frequency lower bound must be "
            "smaller than the upper bound."
        )

    if (
        low is not None
        and convert_quantity_value(
            low,
            frequency_unit,
            u.GHz,
        )
        is None
    ):
        validation_issues.append(
            "Frequency unit is not convertible to GHz."
        )

    resolution_value = parsed.get(
        "resolution_value"
    )
    resolution_unit = parsed.get(
        "resolution_unit"
    )

    if (
        resolution_value is not None
        and convert_quantity_value(
            resolution_value,
            resolution_unit,
            u.MHz,
        )
        is None
    ):
        validation_issues.append(
            "Resolution unit is not convertible to MHz."
        )

    sensitivity_entries = parsed.get(
        "sensitivity_entries",
        [],
    )

    sensitivity_bases = {
        str(entry["basis"]).strip().lower()
        for entry in sensitivity_entries
    }

    for required_basis in {
        "10km/s",
        "native",
    }:
        if required_basis not in sensitivity_bases:
            validation_issues.append(
                f"Missing sensitivity basis: "
                f"{required_basis}."
            )

    for entry in sensitivity_entries:
        try:
            (
                entry["value"]
                * u.Unit(entry["unit"])
            ).to(u.mJy / u.beam)
        except Exception:
            validation_issues.append(
                "Sensitivity unit is not convertible "
                f"to mJy/beam: {entry['unit']}."
            )

    polarization_text = parsed.get(
        "polarization_text"
    )

    polarization_products = (
        set(str(polarization_text).split())
        if polarization_text
        else set()
    )

    unknown_polarization_products = (
        polarization_products
        - VALID_POLARIZATION_PRODUCTS
    )

    if unknown_polarization_products:
        validation_issues.append(
            "Unknown polarization product(s): "
            + ", ".join(
                sorted(
                    unknown_polarization_products
                )
            )
        )

    return list(dict.fromkeys(validation_issues))

In [45]:
parser_contract_records = []


for test_case in parser_boundary_cases:
    parsed = parse_frequency_support_component(
        test_case["component"]
    )

    contract_issues = validate_parsed_component(
        parsed
    )

    parser_contract_records.append(
        {
            "case": test_case["case"],
            "parser_status": parsed["parse_status"],
            "frequency_unit": (
                parsed["frequency_unit"]
            ),
            "resolution_unit": (
                parsed["resolution_unit"]
            ),
            "polarization_text": (
                parsed["polarization_text"]
            ),
            "parser_issues": parsed["issues"],
            "contract_issues": contract_issues,
            "contract_valid": (
                len(contract_issues) == 0
            ),
        }
    )


parser_contract_df = pd.DataFrame(
    parser_contract_records
)


display(parser_contract_df)

,case,parser_status,frequency_unit,resolution_unit,polarization_text,parser_issues,contract_issues,contract_valid
0,standard,PARSED,GHz,kHz,XX YY,[],[],True
1,full_polarization,PARSED,GHz,kHz,XX XY YX YY,[],[],True
2,resolution_in_MHz,PARSED,GHz,MHz,XX YY,[],[],True
3,sensitivity_in_uJy,PARSED,GHz,kHz,XX YY,[],[],True
4,reversed_frequency_range,PARSED,GHz,kHz,XX YY,[],[Frequency lower bound must be smaller than the upper bound.],False
5,missing_native_sensitivity,PARSED,GHz,kHz,XX YY,[],[Missing sensitivity basis: native.],False
6,unknown_polarization_token,PARSED,GHz,kHz,XX YY UNKNOWN,[],[Unknown polarization product(s): UNKNOWN],False
7,invalid_frequency_unit,PARSED,Banana,kHz,XX YY,[],[Frequency unit is not convertible to GHz.],False
8,malformed_frequency_range,FAILED,NaN,kHz,XX YY,[Frequency range could not be parsed.],[Frequency range could not be parsed.],False


## Step 11 — Structured Comparison of Multi-ASDM Signatures

The targeted multi-ASDM Member OUS contained two exact `frequency_support`
strings. This experiment determines whether they represent different spectral
geometries or whether only performance-related values, such as sensitivity,
differ.

A spectral-geometry signature is constructed from frequency bounds, frequency
resolution, and polarization products. Sensitivity values are compared
separately.

In [46]:
def convert_sensitivity_to_mjy_per_beam(
    value: float | None,
    unit_text: str | None,
) -> float | None:
    if value is None or unit_text is None:
        return None

    try:
        quantity = (
            value
            * u.Unit(unit_text)
        )

        return float(
            quantity.to_value(
                u.mJy / u.beam
            )
        )
    except Exception:
        return None

In [47]:
multi_asdm_component_records = []


for context_row in signature_assignment_df.itertuples(
    index=False
):
    components = extract_bracket_components(
        context_row.frequency_support_text
    )

    for component_index, component_text in enumerate(
        components,
        start=1,
    ):
        parsed = parse_frequency_support_component(
            component_text
        )

        low_ghz = convert_quantity_value(
            parsed["frequency_low"],
            parsed["frequency_unit"],
            u.GHz,
        )

        high_ghz = convert_quantity_value(
            parsed["frequency_high"],
            parsed["frequency_unit"],
            u.GHz,
        )

        resolution_mhz = convert_quantity_value(
            parsed["resolution_value"],
            parsed["resolution_unit"],
            u.MHz,
        )

        sensitivity_by_basis = {}

        for entry in parsed["sensitivity_entries"]:
            sensitivity_by_basis[
                entry["basis"]
            ] = convert_sensitivity_to_mjy_per_beam(
                entry["value"],
                entry["unit"],
            )

        multi_asdm_component_records.append(
            {
                "member_ous_uid": (
                    context_row.member_ous_uid
                ),
                "asdm_uid": context_row.asdm_uid,
                "obs_id_source": (
                    context_row.obs_id_source
                ),
                "component_index": component_index,
                "frequency_low_ghz": low_ghz,
                "frequency_high_ghz": high_ghz,
                "resolution_mhz": resolution_mhz,
                "polarization": (
                    parsed["polarization_text"]
                ),
                "sensitivity_10kms_mjy": (
                    sensitivity_by_basis.get(
                        "10km/s"
                    )
                ),
                "sensitivity_native_mjy": (
                    sensitivity_by_basis.get(
                        "native"
                    )
                ),
                "parse_status": (
                    parsed["parse_status"]
                ),
            }
        )


multi_asdm_components_df = pd.DataFrame(
    multi_asdm_component_records
)


display(multi_asdm_components_df)

,member_ous_uid,asdm_uid,obs_id_source,component_index,frequency_low_ghz,frequency_high_ghz,resolution_mhz,polarization,sensitivity_10kms_mjy,sensitivity_native_mjy,parse_status
0,uid://A001/X3833/X1022,uid://A002/X139bfe4/X11e15,SPT2349-56_Core,1,344.45,346.32,7.81201,XX YY,24.6,1.9,PARSED
1,uid://A001/X3833/X1022,uid://A002/X139bfe4/X11e15,SPT2349-56_Core,2,346.32,348.19,7.81201,XX YY,24.5,1.9,PARSED
2,uid://A001/X3833/X1022,uid://A002/X139bfe4/X11e15,SPT2349-56_Core,3,356.68,358.55,7.81201,XX YY,24.3,1.9,PARSED
3,uid://A001/X3833/X1022,uid://A002/X139bfe4/X11e15,SPT2349-56_Core,4,358.38,360.24,7.81201,XX YY,28.8,2.3,PARSED
4,uid://A001/X3833/X1022,uid://A002/X139fbe0/X7472,SPT2349-56_N1-N2,1,344.45,346.32,7.81201,XX YY,21.8,1.7,PARSED
5,uid://A001/X3833/X1022,uid://A002/X139fbe0/X7472,SPT2349-56_N1-N2,2,346.32,348.19,7.81201,XX YY,21.7,1.7,PARSED
6,uid://A001/X3833/X1022,uid://A002/X139fbe0/X7472,SPT2349-56_N1-N2,3,356.68,358.55,7.81201,XX YY,21.5,1.7,PARSED
7,uid://A001/X3833/X1022,uid://A002/X139fbe0/X7472,SPT2349-56_N1-N2,4,358.38,360.24,7.81201,XX YY,25.5,2.0,PARSED


In [48]:
multi_asdm_components_df[
    "spectral_geometry_signature"
] = multi_asdm_components_df.apply(
    lambda row: (
        round(row["frequency_low_ghz"], 9),
        round(row["frequency_high_ghz"], 9),
        round(row["resolution_mhz"], 9),
        row["polarization"],
    ),
    axis=1,
)


multi_asdm_component_comparison_df = (
    multi_asdm_components_df
    .groupby("component_index")
    .agg(
        context_count=("asdm_uid", "size"),
        asdm_count=("asdm_uid", "nunique"),
        source_count=("obs_id_source", "nunique"),
        geometry_signature_count=(
            "spectral_geometry_signature",
            "nunique",
        ),
        sensitivity_10kms_count=(
            "sensitivity_10kms_mjy",
            "nunique",
        ),
        sensitivity_native_count=(
            "sensitivity_native_mjy",
            "nunique",
        ),
    )
    .reset_index()
)


display(multi_asdm_component_comparison_df)

,component_index,context_count,asdm_count,source_count,geometry_signature_count,sensitivity_10kms_count,sensitivity_native_count
0,1,2,2,2,1,2,2
1,2,2,2,2,1,2,2
2,3,2,2,2,1,2,2
3,4,2,2,2,1,2,2


In [49]:
sensitivity_10kms_pivot_df = (
    multi_asdm_components_df
    .pivot(
        index="component_index",
        columns="obs_id_source",
        values="sensitivity_10kms_mjy",
    )
)


sensitivity_native_pivot_df = (
    multi_asdm_components_df
    .pivot(
        index="component_index",
        columns="obs_id_source",
        values="sensitivity_native_mjy",
    )
)


print("10 km/s sensitivity in mJy/beam:")
display(sensitivity_10kms_pivot_df)


print("Native sensitivity in mJy/beam:")
display(sensitivity_native_pivot_df)

10 km/s sensitivity in mJy/beam:


obs_id_source,SPT2349-56_Core,SPT2349-56_N1-N2
component_index,,
1,24.6,21.8
2,24.5,21.7
3,24.3,21.5
4,28.8,25.5


Native sensitivity in mJy/beam:


obs_id_source,SPT2349-56_Core,SPT2349-56_N1-N2
component_index,,
1,1.9,1.7
2,1.9,1.7
3,1.9,1.7
4,2.3,2.0


## Step 12 — Compare Parsed and Row-Level Sensitivity Values

The Archive provides sensitivity information both inside `frequency_support`
and in separate ObsCore columns. This experiment compares the component-level
`@10km/s` sensitivity with the row-level `sensitivity_10kms` field.

The `@native` value is not compared with
`cont_sensitivity_bandwidth`, because native-channel sensitivity and aggregate
continuum sensitivity describe different measurement bases.

In [50]:
member_component_sensitivity_records = []


for component_row in (
    member_parsed_components_df.itertuples(
        index=False
    )
):
    sensitivity_entries = (
        component_row.sensitivity_entries
    )

    for entry in sensitivity_entries:
        normalized_value = (
            convert_sensitivity_to_mjy_per_beam(
                entry["value"],
                entry["unit"],
            )
        )

        member_component_sensitivity_records.append(
            {
                "member_ous_uid": (
                    component_row.member_ous_uid
                ),
                "component_index": (
                    component_row.component_index
                ),
                "sensitivity_basis": (
                    entry["basis"]
                ),
                "original_value": entry["value"],
                "original_unit": entry["unit"],
                "normalized_mjy_per_beam": (
                    normalized_value
                ),
            }
        )


member_component_sensitivity_df = pd.DataFrame(
    member_component_sensitivity_records
)


display(member_component_sensitivity_df.head(20))

,member_ous_uid,component_index,sensitivity_basis,original_value,original_unit,normalized_mjy_per_beam
0,uid://A001/X6f/Xe,1,10km/s,8.8,mJy/beam,8.8000
1,uid://A001/X6f/Xe,1,native,546.7,uJy/beam,0.5467
2,uid://A001/X6f/Xe,2,10km/s,9.3,mJy/beam,9.3000
3,uid://A001/X6f/Xe,2,native,575.8,uJy/beam,0.5758
4,uid://A001/X6f/Xe,3,10km/s,9.1,mJy/beam,9.1000
5,uid://A001/X6f/Xe,3,native,579.6,uJy/beam,0.5796
6,uid://A001/X6f/Xe,4,10km/s,9.0,mJy/beam,9.0000
7,uid://A001/X6f/Xe,4,native,580.1,uJy/beam,0.5801
8,uid://A001/X62/Xb,1,10km/s,2.4,mJy/beam,2.4000
9,uid://A001/X62/Xb,1,native,293.7,uJy/beam,0.2937


In [51]:
component_sensitivity_wide_df = (
    member_component_sensitivity_df
    .pivot_table(
        index=[
            "member_ous_uid",
            "component_index",
        ],
        columns="sensitivity_basis",
        values="normalized_mjy_per_beam",
        aggfunc="first",
    )
    .reset_index()
)


component_sensitivity_wide_df.columns.name = None


component_sensitivity_wide_df = (
    component_sensitivity_wide_df.rename(
        columns={
            "10km/s": (
                "parsed_sensitivity_10kms_mjy"
            ),
            "native": (
                "parsed_sensitivity_native_mjy"
            ),
        }
    )
)


display(component_sensitivity_wide_df.head(20))

,member_ous_uid,component_index,parsed_sensitivity_10kms_mjy,parsed_sensitivity_native_mjy
0,uid://A001/X11d/Xf,1,2.7000,0.1689
1,uid://A001/X11d/Xf,2,2.6000,0.1617
2,uid://A001/X11d/Xf,3,2.7000,0.1729
3,uid://A001/X11d/Xf,4,2.7000,0.1731
4,uid://A001/X11e/X5,1,0.7616,0.0465
5,uid://A001/X11e/X5,2,0.7372,0.0453
6,uid://A001/X11e/X5,3,0.7658,0.0484
7,uid://A001/X11e/X5,4,0.7634,0.0485
8,uid://A001/X11f/X6,1,5.4000,0.3259
9,uid://A001/X11f/X6,2,5.4000,1.9000


In [52]:
logical_spw_sensitivity_df = (
    archive_sample_df
    .groupby(
        [
            "member_ous_uid",
            "spw_identifier",
        ],
        dropna=False,
    )
    .agg(
        row_count=("obs_id", "size"),
        source_count=("obs_id_source", "nunique"),
        row_sensitivity_10kms_median=(
            "sensitivity_10kms",
            "median",
        ),
        row_sensitivity_10kms_min=(
            "sensitivity_10kms",
            "min",
        ),
        row_sensitivity_10kms_max=(
            "sensitivity_10kms",
            "max",
        ),
        row_cont_sensitivity_median=(
            "cont_sensitivity_bandwidth",
            "median",
        ),
        row_cont_sensitivity_min=(
            "cont_sensitivity_bandwidth",
            "min",
        ),
        row_cont_sensitivity_max=(
            "cont_sensitivity_bandwidth",
            "max",
        ),
    )
    .reset_index()
)


display(logical_spw_sensitivity_df.head(20))

,member_ous_uid,spw_identifier,row_count,source_count,row_sensitivity_10kms_median,row_sensitivity_10kms_min,row_sensitivity_10kms_max,row_cont_sensitivity_median,row_cont_sensitivity_min,row_cont_sensitivity_max
0,uid://A001/X11d/Xf,11,20,20,2.566399,2.566399,2.566400,0.084469,0.084469,0.084469
1,uid://A001/X11d/Xf,19,20,20,2.666483,2.666483,2.666484,0.084469,0.084469,0.084469
2,uid://A001/X11d/Xf,21,20,20,2.658918,2.658918,2.658919,0.084469,0.084469,0.084469
3,uid://A001/X11d/Xf,9,20,20,2.691530,2.691529,2.691530,0.084469,0.084469,0.084469
4,uid://A001/X11e/X5,11,1,1,0.737197,0.737197,0.737197,0.023564,0.023564,0.023564
5,uid://A001/X11e/X5,19,1,1,0.765802,0.765802,0.765802,0.023564,0.023564,0.023564
6,uid://A001/X11e/X5,21,1,1,0.763438,0.763438,0.763438,0.023564,0.023564,0.023564
7,uid://A001/X11e/X5,9,1,1,0.761617,0.761617,0.761617,0.023564,0.023564,0.023564
8,uid://A001/X11f/X6,17,1,1,5.389245,5.389245,5.389245,0.224787,0.224787,0.224787
9,uid://A001/X11f/X6,19,1,1,5.381165,5.381165,5.381165,0.224787,0.224787,0.224787


In [53]:
sensitivity_comparison_df = (
    spw_interval_assignment_df[
        [
            "member_ous_uid",
            "spw_identifier",
            "component_index",
        ]
    ]
    .merge(
        component_sensitivity_wide_df,
        on=[
            "member_ous_uid",
            "component_index",
        ],
        how="left",
        validate="one_to_one",
    )
    .merge(
        logical_spw_sensitivity_df,
        on=[
            "member_ous_uid",
            "spw_identifier",
        ],
        how="left",
        validate="one_to_one",
    )
)


sensitivity_comparison_df[
    "sensitivity_10kms_difference_mjy"
] = (
    sensitivity_comparison_df[
        "row_sensitivity_10kms_median"
    ]
    - sensitivity_comparison_df[
        "parsed_sensitivity_10kms_mjy"
    ]
)


sensitivity_comparison_df[
    "absolute_sensitivity_10kms_difference_mjy"
] = sensitivity_comparison_df[
    "sensitivity_10kms_difference_mjy"
].abs()


display(
    sensitivity_comparison_df[
        [
            "member_ous_uid",
            "spw_identifier",
            "component_index",
            "parsed_sensitivity_10kms_mjy",
            "row_sensitivity_10kms_median",
            "sensitivity_10kms_difference_mjy",
            "parsed_sensitivity_native_mjy",
            "row_cont_sensitivity_median",
        ]
    ].head(30)
)

,member_ous_uid,spw_identifier,component_index,parsed_sensitivity_10kms_mjy,row_sensitivity_10kms_median,sensitivity_10kms_difference_mjy,parsed_sensitivity_native_mjy,row_cont_sensitivity_median
0,uid://A001/X11d/Xf,11,2,2.6000,2.566399,-3.360074e-02,0.1617,0.084469
1,uid://A001/X11d/Xf,19,3,2.7000,2.666483,-3.351685e-02,0.1729,0.084469
2,uid://A001/X11d/Xf,21,4,2.7000,2.658918,-4.108157e-02,0.1731,0.084469
3,uid://A001/X11d/Xf,9,1,2.7000,2.691530,-8.470355e-03,0.1689,0.084469
4,uid://A001/X11e/X5,11,2,0.7372,0.737197,-2.512843e-06,0.0453,0.023564
5,uid://A001/X11e/X5,19,3,0.7658,0.765802,1.893607e-06,0.0484,0.023564
6,uid://A001/X11e/X5,21,4,0.7634,0.763438,3.845025e-05,0.0485,0.023564
7,uid://A001/X11e/X5,9,1,0.7616,0.761617,1.672003e-05,0.0465,0.023564
8,uid://A001/X11f/X6,17,2,5.4000,5.389245,-1.075492e-02,1.9000,0.224787
9,uid://A001/X11f/X6,19,3,5.4000,5.381165,-1.883521e-02,1.9000,0.224787


In [54]:
display(
    sensitivity_comparison_df[
        [
            "absolute_sensitivity_10kms_difference_mjy",
        ]
    ].describe(
        percentiles=[
            0.5,
            0.9,
            0.95,
            0.99,
        ]
    )
)


display(
    sensitivity_comparison_df
    .sort_values(
        "absolute_sensitivity_10kms_difference_mjy",
        ascending=False,
    )
    [
        [
            "member_ous_uid",
            "spw_identifier",
            "component_index",
            "parsed_sensitivity_10kms_mjy",
            "row_sensitivity_10kms_median",
            "absolute_sensitivity_10kms_difference_mjy",
            "source_count",
        ]
    ]
    .head(20)
)

,absolute_sensitivity_10kms_difference_mjy
count,9.500000e+01
mean,1.672624e-02
std,1.544941e-02
min,5.996385e-08
50%,1.670685e-02
90%,3.671517e-02
95%,4.244555e-02
99%,4.482056e-02
max,4.722359e-02


,member_ous_uid,spw_identifier,component_index,parsed_sensitivity_10kms_mjy,row_sensitivity_10kms_median,absolute_sensitivity_10kms_difference_mjy,source_count
57,uid://A001/X137/Xd,23,2,8.6,8.552776,0.047224,1
91,uid://A001/X6f/Xe,17,3,9.1,9.055333,0.044667,1
18,uid://A001/X11f/X8,24,5,24.8,24.756196,0.043804,1
56,uid://A001/X137/Xd,21,1,8.6,8.556242,0.043758,1
60,uid://A001/X140/X6,15,4,1.5,1.457178,0.042822,1
26,uid://A001/X12a/X9,21,1,1.6,1.642284,0.042284,1
19,uid://A001/X11f/X8,26,6,23.9,23.942049,0.042049,1
27,uid://A001/X12a/X9,23,2,1.6,1.641318,0.041318,1
2,uid://A001/X11d/Xf,21,4,2.7,2.658918,0.041082,20
21,uid://A001/X123/Xc,19,3,10.2,10.237001,0.037001,1


## Step 13 — Measure Missing Metadata in the Archive

The initial purposive sample contained no missing `frequency_support` values.
This experiment measures the prevalence of missing frequency and sensitivity
metadata across all science-observation rows.

Missing metadata must remain distinct from zero sensitivity, an empty spectral
setup, or a failed duplication criterion.

In [55]:
missingness_conditions = {
    "all_science_rows": "1 = 1",
    "frequency_support_null": (
        "frequency_support IS NULL"
    ),
    "sensitivity_10kms_null": (
        "sensitivity_10kms IS NULL"
    ),
    "continuum_sensitivity_null": (
        "cont_sensitivity_bandwidth IS NULL"
    ),
}


archive_missingness_records = []


for label, condition in (
    missingness_conditions.items()
):
    missingness_query = f"""
SELECT COUNT(*) AS total_rows
FROM ivoa.obscore
WHERE science_observation = 'T'
AND {condition}
"""

    result_table = run_tap_query(
        missingness_query,
        maxrec=10,
        label=label,
    )

    archive_missingness_records.append(
        {
            "category": label,
            "row_count": int(
                result_table["total_rows"][0]
            ),
        }
    )


archive_missingness_df = pd.DataFrame(
    archive_missingness_records
)


total_science_rows = int(
    archive_missingness_df.loc[
        archive_missingness_df["category"]
        == "all_science_rows",
        "row_count",
    ].iloc[0]
)


archive_missingness_df[
    "fraction_of_science_rows"
] = (
    archive_missingness_df["row_count"]
    / total_science_rows
)


archive_missingness_df[
    "percentage_of_science_rows"
] = (
    archive_missingness_df[
        "fraction_of_science_rows"
    ]
    * 100
)


display(archive_missingness_df)

Running: all_science_rows
Retrieved 1 rows in 6.7 seconds.
Running: frequency_support_null
Retrieved 1 rows in 1.7 seconds.
Running: sensitivity_10kms_null
Retrieved 1 rows in 5.8 seconds.
Running: continuum_sensitivity_null
Retrieved 1 rows in 1.7 seconds.


,category,row_count,fraction_of_science_rows,percentage_of_science_rows
0,all_science_rows,442506,1.0,100.0
1,frequency_support_null,0,0.0,0.0
2,sensitivity_10kms_null,0,0.0,0.0
3,continuum_sensitivity_null,0,0.0,0.0


In [56]:
missing_frequency_support_query = """
SELECT TOP 50
    proposal_id,
    member_ous_uid,
    asdm_uid,
    obs_id,
    target_name,
    frequency,
    bandwidth,
    sensitivity_10kms,
    cont_sensitivity_bandwidth,
    is_mosaic
FROM ivoa.obscore
WHERE science_observation = 'T'
AND frequency_support IS NULL
"""


missing_frequency_support_table = run_tap_query(
    missing_frequency_support_query,
    maxrec=50,
    label="Inspect rows without frequency_support",
)


missing_frequency_support_df = (
    missing_frequency_support_table
    .to_pandas()
    .copy()
)


display(missing_frequency_support_df.head(50))

Running: Inspect rows without frequency_support
Retrieved 0 rows in 1.5 seconds.


,proposal_id,member_ous_uid,asdm_uid,obs_id,target_name,frequency,bandwidth,sensitivity_10kms,cont_sensitivity_bandwidth,is_mosaic


## Step 14 — Validate the Grammar Across ALMA Frequency Bands

The first sample was selected by a limited number of frequency strata. This
experiment expands the grammar validation across ALMA Bands 3–10.

The sample is purposive rather than statistically representative. Its purpose
is to expose alternative units, component counts, sensitivity formats,
polarization products, and parser failures.

In [57]:
alma_band_ranges = [
    {
        "band": "Band 3",
        "frequency_min_ghz": 84.0,
        "frequency_max_ghz": 116.0,
    },
    {
        "band": "Band 4",
        "frequency_min_ghz": 125.0,
        "frequency_max_ghz": 163.0,
    },
    {
        "band": "Band 5",
        "frequency_min_ghz": 163.0,
        "frequency_max_ghz": 211.0,
    },
    {
        "band": "Band 6",
        "frequency_min_ghz": 211.0,
        "frequency_max_ghz": 275.0,
    },
    {
        "band": "Band 7",
        "frequency_min_ghz": 275.0,
        "frequency_max_ghz": 373.0,
    },
    {
        "band": "Band 8",
        "frequency_min_ghz": 385.0,
        "frequency_max_ghz": 500.0,
    },
    {
        "band": "Band 9",
        "frequency_min_ghz": 602.0,
        "frequency_max_ghz": 720.0,
    },
    {
        "band": "Band 10",
        "frequency_min_ghz": 787.0,
        "frequency_max_ghz": 950.0,
    },
]

In [58]:
band_signature_tables = []


for band_definition in alma_band_ranges:
    band_query = f"""
SELECT DISTINCT TOP 10
    member_ous_uid,
    proposal_id,
    frequency_support
FROM ivoa.obscore
WHERE science_observation = 'T'
AND member_ous_uid IS NOT NULL
AND frequency_support IS NOT NULL
AND frequency >= {
    band_definition["frequency_min_ghz"]
}
AND frequency < {
    band_definition["frequency_max_ghz"]
}
"""

    band_table = run_tap_query(
        band_query,
        maxrec=10,
        label=band_definition["band"],
    )

    band_df = band_table.to_pandas().copy()

    if band_df.empty:
        print(
            "No records returned for",
            band_definition["band"],
        )
        continue

    band_df["sample_band"] = (
        band_definition["band"]
    )

    band_signature_tables.append(band_df)

Running: Band 3
Retrieved 10 rows in 7.6 seconds.
Running: Band 4
Retrieved 10 rows in 5.1 seconds.
Running: Band 5
Retrieved 10 rows in 5.1 seconds.
Running: Band 6
Retrieved 10 rows in 5.7 seconds.
Running: Band 7
Retrieved 10 rows in 5.5 seconds.
Running: Band 8
Retrieved 10 rows in 4.6 seconds.
Running: Band 9
Retrieved 10 rows in 4.5 seconds.
Running: Band 10
Retrieved 10 rows in 6.1 seconds.


In [60]:
expanded_band_signature_df = pd.concat(
    band_signature_tables,
    ignore_index=True,
)


expanded_band_signature_df[
    "frequency_support_text"
] = expanded_band_signature_df[
    "frequency_support"
].map(
    normalize_frequency_support_text
)


expanded_band_component_records = []


for row in expanded_band_signature_df.itertuples(
    index=False
):
    components = extract_bracket_components(
        row.frequency_support_text
    )

    if not components:
        expanded_band_component_records.append(
            {
                "sample_band": row.sample_band,
                "member_ous_uid": row.member_ous_uid,
                "proposal_id": row.proposal_id,
                "component_index": None,
                "parse_status": "FAILED",
                "validation_status": "INVALID",
                "validation_issues": [
                    "No bracketed component found."
                ],
            }
        )
        continue

    for component_index, component_text in enumerate(
        components,
        start=1,
    ):
        parsed = parse_frequency_support_component(
            component_text
        )

        validation_issues = (
            validate_parsed_component(parsed)
        )

        expanded_band_component_records.append(
            {
                "sample_band": row.sample_band,
                "member_ous_uid": row.member_ous_uid,
                "proposal_id": row.proposal_id,
                "component_index": component_index,
                "component_text": component_text,
                "frequency_unit": (
                    parsed["frequency_unit"]
                ),
                "resolution_unit": (
                    parsed["resolution_unit"]
                ),
                "polarization": (
                    parsed["polarization_text"]
                ),
                "parse_status": (
                    parsed["parse_status"]
                ),
                "validation_status": (
                    "VALID"
                    if not validation_issues
                    else "INVALID"
                ),
                "validation_issues": (
                    validation_issues
                ),
            }
        )


expanded_band_components_df = pd.DataFrame(
    expanded_band_component_records
)

In [61]:
band_validation_summary_df = (
    expanded_band_components_df
    .groupby("sample_band")
    .agg(
        member_count=("member_ous_uid", "nunique"),
        component_count=("component_index", "count"),
        valid_component_count=(
            "validation_status",
            lambda values: (
                values == "VALID"
            ).sum(),
        ),
        invalid_component_count=(
            "validation_status",
            lambda values: (
                values == "INVALID"
            ).sum(),
        ),
        frequency_unit_count=(
            "frequency_unit",
            "nunique",
        ),
        resolution_unit_count=(
            "resolution_unit",
            "nunique",
        ),
        polarization_format_count=(
            "polarization",
            "nunique",
        ),
    )
    .reset_index()
)


display(band_validation_summary_df)


print("Frequency units:")
display(
    expanded_band_components_df[
        [
            "sample_band",
            "frequency_unit",
        ]
    ]
    .value_counts()
    .rename("component_count")
    .reset_index()
)


print("Resolution units:")
display(
    expanded_band_components_df[
        [
            "sample_band",
            "resolution_unit",
        ]
    ]
    .value_counts()
    .rename("component_count")
    .reset_index()
)


print("Components requiring review:")
display(
    expanded_band_components_df[
        expanded_band_components_df[
            "validation_status"
        ]
        == "INVALID"
    ].head(100)
)

,sample_band,member_count,component_count,valid_component_count,invalid_component_count,frequency_unit_count,resolution_unit_count,polarization_format_count
0,Band 10,10,66,66,0,1,1,1
1,Band 3,10,52,52,0,1,1,1
2,Band 4,10,40,40,0,1,1,1
3,Band 5,9,56,56,0,1,1,1
4,Band 6,10,49,49,0,1,1,1
5,Band 7,9,48,48,0,1,1,1
6,Band 8,10,44,44,0,1,1,1
7,Band 9,10,46,46,0,1,1,1


Frequency units:


,sample_band,frequency_unit,component_count
0,Band 10,GHz,66
1,Band 5,GHz,56
2,Band 3,GHz,52
3,Band 6,GHz,49
4,Band 7,GHz,48
5,Band 9,GHz,46
6,Band 8,GHz,44
7,Band 4,GHz,40


Resolution units:


,sample_band,resolution_unit,component_count
0,Band 10,kHz,66
1,Band 5,kHz,56
2,Band 3,kHz,52
3,Band 6,kHz,49
4,Band 7,kHz,48
5,Band 9,kHz,46
6,Band 8,kHz,44
7,Band 4,kHz,40


Components requiring review:


,sample_band,member_ous_uid,proposal_id,component_index,component_text,frequency_unit,resolution_unit,polarization,parse_status,validation_status,validation_issues


## Step 15 — Search for Additional Multi-ASDM Ownership Cases

The first targeted multi-ASDM example paired each ASDM with a different source,
so source effects and execution effects could not be separated.

This experiment searches for additional recent Member OUS datasets containing
multiple ASDM UIDs, then identifies source labels appearing in more than one
ASDM execution.

In [62]:
multi_asdm_candidate_query = """
SELECT TOP 10
    member_ous_uid,
    COUNT(DISTINCT asdm_uid) AS asdm_count
FROM ivoa.obscore
WHERE science_observation = 'T'
AND member_ous_uid IS NOT NULL
AND asdm_uid IS NOT NULL
AND proposal_id LIKE '202%'
GROUP BY member_ous_uid
HAVING COUNT(DISTINCT asdm_uid) > 1
ORDER BY asdm_count DESC
"""


multi_asdm_candidate_table = run_tap_query(
    multi_asdm_candidate_query,
    maxrec=10,
    label="Search for recent multi-ASDM Members",
)


additional_multi_asdm_candidates_df = (
    multi_asdm_candidate_table
    .to_pandas()
    .copy()
)


display(additional_multi_asdm_candidates_df)

Running: Search for recent multi-ASDM Members
Retrieved 10 rows in 3.0 seconds.


,member_ous_uid,asdm_count
0,uid://A001/X37fb/X35d,3
1,uid://A001/X37fb/X361,3
2,uid://A001/X3577/X1df,2
3,uid://A001/X15a9/X9f9,2
4,uid://A001/X3621/X233d,2
5,uid://A001/X3621/X2a50,2
6,uid://A001/X2df7/X7be,2
7,uid://A001/X3833/X65b6,2
8,uid://A001/X3788/X540b,2
9,uid://A001/X3788/X623c,2


In [63]:
selected_multi_asdm_member_uids = (
    additional_multi_asdm_candidates_df[
        "member_ous_uid"
    ]
    .dropna()
    .astype(str)
    .head(5)
    .tolist()
)


if not selected_multi_asdm_member_uids:
    print("No additional multi-ASDM candidates found.")

else:
    selected_multi_asdm_sql = ",\n    ".join(
        quote_adql_string(member_uid)
        for member_uid
        in selected_multi_asdm_member_uids
    )

    additional_multi_asdm_count_query = f"""
SELECT COUNT(*) AS total_rows
FROM ivoa.obscore
WHERE science_observation = 'T'
AND member_ous_uid IN (
    {selected_multi_asdm_sql}
)
"""

    additional_count_table = run_tap_query(
        additional_multi_asdm_count_query,
        maxrec=10,
        label="Count additional multi-ASDM rows",
    )

    additional_multi_asdm_total_rows = int(
        additional_count_table[
            "total_rows"
        ][0]
    )

    print(
        "Expected additional rows:",
        additional_multi_asdm_total_rows,
    )

Running: Count additional multi-ASDM rows
Retrieved 1 rows in 2.0 seconds.
Expected additional rows: 54


In [64]:
if selected_multi_asdm_member_uids:
    if additional_multi_asdm_total_rows > 5000:
        raise RuntimeError(
            "The multi-ASDM validation sample "
            "exceeds 5,000 rows. Reduce the "
            "candidate count before downloading."
        )

    additional_multi_asdm_query = f"""
SELECT
    {column_sql}
FROM ivoa.obscore
WHERE science_observation = 'T'
AND member_ous_uid IN (
    {selected_multi_asdm_sql}
)
"""

    additional_multi_asdm_table = run_tap_query(
        additional_multi_asdm_query,
        maxrec=max(
            additional_multi_asdm_total_rows,
            1,
        ),
        label="Retrieve additional multi-ASDM rows",
    )

    additional_multi_asdm_df = (
        additional_multi_asdm_table
        .to_pandas()
        .copy()
    )

    parsed_additional_obs_ids = (
        additional_multi_asdm_df[
            "obs_id"
        ].map(parse_obs_id)
    )

    additional_multi_asdm_df[
        "obs_id_source"
    ] = [
        value[0]
        for value in parsed_additional_obs_ids
    ]

    additional_multi_asdm_df[
        "spw_identifier"
    ] = [
        value[1]
        for value in parsed_additional_obs_ids
    ]

    additional_multi_asdm_df[
        "frequency_support_text"
    ] = additional_multi_asdm_df[
        "frequency_support"
    ].map(
        normalize_frequency_support_text
    )

    print(
        "Retrieved rows:",
        len(additional_multi_asdm_df),
    )

Running: Retrieve additional multi-ASDM rows
Retrieved 54 rows in 1.9 seconds.
Retrieved rows: 54


In [65]:
if selected_multi_asdm_member_uids:
    source_execution_relationship_df = (
        additional_multi_asdm_df
        .groupby(
            [
                "member_ous_uid",
                "obs_id_source",
            ],
            dropna=False,
        )
        .agg(
            archive_rows=("obs_id", "size"),
            asdm_count=("asdm_uid", "nunique"),
            spw_count=(
                "spw_identifier",
                "nunique",
            ),
            exact_signature_count=(
                "frequency_support_text",
                "nunique",
            ),
        )
        .reset_index()
    )

    same_source_multiple_asdm_df = (
        source_execution_relationship_df[
            source_execution_relationship_df[
                "asdm_count"
            ]
            > 1
        ]
        .sort_values(
            [
                "asdm_count",
                "exact_signature_count",
            ],
            ascending=False,
        )
    )

    print(
        "Source contexts appearing in "
        "multiple ASDM executions:",
        len(same_source_multiple_asdm_df),
    )

    display(same_source_multiple_asdm_df)

Source contexts appearing in multiple ASDM executions: 0


,member_ous_uid,obs_id_source,archive_rows,asdm_count,spw_count,exact_signature_count


## Step 16 — Investigate Repeated Members in the Band Sample

The Band 5 and Band 7 samples each returned ten distinct query records but
only nine unique Member OUS identifiers. This experiment determines whether
the repeated Members contain different proposal identifiers, different exact
frequency-support strings, different spectral geometries, or only different
sensitivity estimates.

This distinction is important because exact raw-string inequality does not
necessarily imply a different correlator setup.

In [66]:
repeated_band_member_summary_df = (
    expanded_band_signature_df
    .groupby(
        [
            "sample_band",
            "member_ous_uid",
        ],
        dropna=False,
    )
    .agg(
        returned_record_count=(
            "frequency_support_text",
            "size",
        ),
        proposal_count=(
            "proposal_id",
            "nunique",
        ),
        exact_signature_count=(
            "frequency_support_text",
            "nunique",
        ),
    )
    .reset_index()
)


repeated_band_members_df = (
    repeated_band_member_summary_df[
        repeated_band_member_summary_df[
            "returned_record_count"
        ]
        > 1
    ]
    .sort_values(
        [
            "sample_band",
            "member_ous_uid",
        ]
    )
    .reset_index(drop=True)
)


display(repeated_band_members_df)

,sample_band,member_ous_uid,returned_record_count,proposal_count,exact_signature_count
0,Band 5,uid://A001/X1359/Xb,2,1,2
1,Band 7,uid://A001/X62/X6,2,1,2


In [67]:
repeated_band_member_details_df = (
    expanded_band_signature_df
    .merge(
        repeated_band_members_df[
            [
                "sample_band",
                "member_ous_uid",
            ]
        ],
        on=[
            "sample_band",
            "member_ous_uid",
        ],
        how="inner",
        validate="many_to_one",
    )
    [
        [
            "sample_band",
            "member_ous_uid",
            "proposal_id",
            "frequency_support_text",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "sample_band",
            "member_ous_uid",
        ]
    )
    .reset_index(drop=True)
)


display(repeated_band_member_details_df)

,sample_band,member_ous_uid,proposal_id,frequency_support_text
0,Band 5,uid://A001/X1359/Xb,2018.1.00375.S,"[167.88..167.94GHz,122.07kHz,10.7mJy/beam@10km/s,3.2mJy/beam@native, XX YY] U [168.73..168.79GHz,122.07kHz,11.1mJy/beam@10km/s,3.3mJy/beam@native, XX YY] U [168.78..168.84GHz,1..."
1,Band 5,uid://A001/X1359/Xb,2018.1.00375.S,"[167.88..167.94GHz,122.07kHz,10.8mJy/beam@10km/s,3.2mJy/beam@native, XX YY] U [168.73..168.79GHz,122.07kHz,11.2mJy/beam@10km/s,3.4mJy/beam@native, XX YY] U [168.78..168.84GHz,1..."
2,Band 7,uid://A001/X62/X6,2011.0.00780.S,"[337.02..339.00GHz,31250.00kHz,3.1mJy/beam@10km/s,235.7uJy/beam@native, XX YY] U [338.95..340.94GHz,31250.00kHz,3mJy/beam@10km/s,227.9uJy/beam@native, XX YY] U [349.02..351.00G..."
3,Band 7,uid://A001/X62/X6,2011.0.00780.S,"[337.02..339.01GHz,31250.00kHz,2.2mJy/beam@10km/s,162.2uJy/beam@native, XX YY] U [338.96..340.94GHz,31250.00kHz,2.1mJy/beam@10km/s,156.9uJy/beam@native, XX YY] U [349.02..351.0..."


In [68]:
repeated_signature_records = []


for row in repeated_band_member_details_df.itertuples(
    index=False
):
    geometry_components = []
    sensitivity_components = []

    for component_text in extract_bracket_components(
        row.frequency_support_text
    ):
        parsed = parse_frequency_support_component(
            component_text
        )

        low_ghz = convert_quantity_value(
            parsed["frequency_low"],
            parsed["frequency_unit"],
            u.GHz,
        )

        high_ghz = convert_quantity_value(
            parsed["frequency_high"],
            parsed["frequency_unit"],
            u.GHz,
        )

        resolution_mhz = convert_quantity_value(
            parsed["resolution_value"],
            parsed["resolution_unit"],
            u.MHz,
        )

        geometry_components.append(
            (
                round(low_ghz, 9),
                round(high_ghz, 9),
                round(resolution_mhz, 9),
                parsed["polarization_text"],
            )
        )

        component_sensitivities = []

        for entry in parsed["sensitivity_entries"]:
            component_sensitivities.append(
                (
                    entry["basis"],
                    convert_sensitivity_to_mjy_per_beam(
                        entry["value"],
                        entry["unit"],
                    ),
                )
            )

        sensitivity_components.append(
            tuple(component_sensitivities)
        )

    repeated_signature_records.append(
        {
            "sample_band": row.sample_band,
            "member_ous_uid": row.member_ous_uid,
            "proposal_id": row.proposal_id,
            "exact_signature": (
                row.frequency_support_text
            ),
            "geometry_signature": tuple(
                geometry_components
            ),
            "sensitivity_signature": tuple(
                sensitivity_components
            ),
        }
    )


repeated_signature_comparison_df = pd.DataFrame(
    repeated_signature_records
)


display(repeated_signature_comparison_df)

,sample_band,member_ous_uid,proposal_id,exact_signature,geometry_signature,sensitivity_signature
0,Band 5,uid://A001/X1359/Xb,2018.1.00375.S,"[167.88..167.94GHz,122.07kHz,10.7mJy/beam@10km/s,3.2mJy/beam@native, XX YY] U [168.73..168.79GHz,122.07kHz,11.1mJy/beam@10km/s,3.3mJy/beam@native, XX YY] U [168.78..168.84GHz,1...","((167.88, 167.94, 0.12207, XX YY), (168.73, 168.79, 0.12207, XX YY), (168.78, 168.84, 0.12207, XX YY), (169.08, 169.14, 0.12207, XX YY), (169.69, 169.81, 0.12207, XX YY), (169....","(((10km/s, 10.7), (native, 3.2)), ((10km/s, 11.1), (native, 3.3)), ((10km/s, 11.1), (native, 3.3)), ((10km/s, 11.1), (native, 3.3)), ((10km/s, 10.1), (native, 2.2)), ((10km/s, ..."
1,Band 5,uid://A001/X1359/Xb,2018.1.00375.S,"[167.88..167.94GHz,122.07kHz,10.8mJy/beam@10km/s,3.2mJy/beam@native, XX YY] U [168.73..168.79GHz,122.07kHz,11.2mJy/beam@10km/s,3.4mJy/beam@native, XX YY] U [168.78..168.84GHz,1...","((167.88, 167.94, 0.12207, XX YY), (168.73, 168.79, 0.12207, XX YY), (168.78, 168.84, 0.12207, XX YY), (169.08, 169.14, 0.12207, XX YY), (169.69, 169.81, 0.12207, XX YY), (169....","(((10km/s, 10.8), (native, 3.2)), ((10km/s, 11.2), (native, 3.4)), ((10km/s, 11.3), (native, 3.4)), ((10km/s, 11.2), (native, 3.4)), ((10km/s, 10.2), (native, 2.2)), ((10km/s, ..."
2,Band 7,uid://A001/X62/X6,2011.0.00780.S,"[337.02..339.00GHz,31250.00kHz,3.1mJy/beam@10km/s,235.7uJy/beam@native, XX YY] U [338.95..340.94GHz,31250.00kHz,3mJy/beam@10km/s,227.9uJy/beam@native, XX YY] U [349.02..351.00G...","((337.02, 339.0, 31.25, XX YY), (338.95, 340.94, 31.25, XX YY), (349.02, 351.0, 31.25, XX YY), (351.02, 353.0, 31.25, XX YY))","(((10km/s, 3.1), (native, 0.2357)), ((10km/s, 3.0), (native, 0.22790000000000002)), ((10km/s, 3.3), (native, 0.2494)), ((10km/s, 3.8), (native, 0.29410000000000003)))"
3,Band 7,uid://A001/X62/X6,2011.0.00780.S,"[337.02..339.01GHz,31250.00kHz,2.2mJy/beam@10km/s,162.2uJy/beam@native, XX YY] U [338.96..340.94GHz,31250.00kHz,2.1mJy/beam@10km/s,156.9uJy/beam@native, XX YY] U [349.02..351.0...","((337.02, 339.01, 31.25, XX YY), (338.96, 340.94, 31.25, XX YY), (349.02, 351.01, 31.25, XX YY), (351.02, 353.01, 31.25, XX YY))","(((10km/s, 2.2), (native, 0.16219999999999998)), ((10km/s, 2.1), (native, 0.1569)), ((10km/s, 2.2), (native, 0.1691)), ((10km/s, 2.6), (native, 0.1975)))"


In [69]:
repeated_signature_classification_df = (
    repeated_signature_comparison_df
    .groupby(
        [
            "sample_band",
            "member_ous_uid",
        ]
    )
    .agg(
        proposal_count=(
            "proposal_id",
            "nunique",
        ),
        exact_signature_count=(
            "exact_signature",
            "nunique",
        ),
        geometry_signature_count=(
            "geometry_signature",
            "nunique",
        ),
        sensitivity_signature_count=(
            "sensitivity_signature",
            "nunique",
        ),
    )
    .reset_index()
)


repeated_signature_classification_df[
    "preliminary_interpretation"
] = repeated_signature_classification_df.apply(
    lambda row: (
        "Different spectral geometry"
        if row["geometry_signature_count"] > 1
        else (
            "Same geometry, different sensitivity"
            if row["sensitivity_signature_count"] > 1
            else "Equivalent parsed configuration"
        )
    ),
    axis=1,
)


display(repeated_signature_classification_df)

,sample_band,member_ous_uid,proposal_count,exact_signature_count,geometry_signature_count,sensitivity_signature_count,preliminary_interpretation
0,Band 5,uid://A001/X1359/Xb,1,2,1,2,"Same geometry, different sensitivity"
1,Band 7,uid://A001/X62/X6,1,2,2,2,Different spectral geometry


## Step 17 — Validate Non-Null Metadata Values

The Archive-wide query found no SQL NULL values in `frequency_support`,
`sensitivity_10kms`, or `cont_sensitivity_bandwidth`. SQL non-null status does
not guarantee that a value is non-blank, finite, positive, or scientifically
usable.

This experiment searches for non-null but structurally or numerically invalid
values.

In [70]:
semantic_validity_conditions = {
    "blank_frequency_support": (
        "frequency_support = ''"
    ),
    "nonpositive_frequency": (
        "frequency <= 0"
    ),
    "nonpositive_bandwidth": (
        "bandwidth <= 0"
    ),
    "nonpositive_sensitivity_10kms": (
        "sensitivity_10kms <= 0"
    ),
    "nonpositive_continuum_sensitivity": (
        "cont_sensitivity_bandwidth <= 0"
    ),
}


semantic_validity_records = []


for label, condition in (
    semantic_validity_conditions.items()
):
    validity_query = f"""
SELECT COUNT(*) AS total_rows
FROM ivoa.obscore
WHERE science_observation = 'T'
AND {condition}
"""

    result_table = run_tap_query(
        validity_query,
        maxrec=10,
        label=label,
    )

    semantic_validity_records.append(
        {
            "category": label,
            "row_count": int(
                result_table["total_rows"][0]
            ),
        }
    )


semantic_validity_df = pd.DataFrame(
    semantic_validity_records
)


semantic_validity_df[
    "percentage_of_science_rows"
] = (
    semantic_validity_df["row_count"]
    / total_science_rows
    * 100
)


display(semantic_validity_df)

Running: blank_frequency_support
Retrieved 1 rows in 1.2 seconds.
Running: nonpositive_frequency
Retrieved 1 rows in 6.1 seconds.
Running: nonpositive_bandwidth
Retrieved 1 rows in 4.0 seconds.
Running: nonpositive_sensitivity_10kms
Retrieved 1 rows in 4.5 seconds.
Running: nonpositive_continuum_sensitivity
Retrieved 1 rows in 1.8 seconds.


,category,row_count,percentage_of_science_rows
0,blank_frequency_support,0,0.0
1,nonpositive_frequency,0,0.0
2,nonpositive_bandwidth,0,0.0
3,nonpositive_sensitivity_10kms,0,0.0
4,nonpositive_continuum_sensitivity,0,0.0


In [71]:
local_validation_frames = [
    archive_sample_df.copy(),
    multi_asdm_df.copy(),
]


if (
    "additional_multi_asdm_df"
    in globals()
):
    local_validation_frames.append(
        additional_multi_asdm_df.copy()
    )


combined_local_validation_df = pd.concat(
    local_validation_frames,
    ignore_index=True,
)


numeric_validation_columns = [
    "frequency",
    "bandwidth",
    "sensitivity_10kms",
    "cont_sensitivity_bandwidth",
    "spatial_resolution",
    "t_min",
    "t_max",
]


local_numeric_validity_records = []


for column_name in numeric_validation_columns:
    numeric_values = pd.to_numeric(
        combined_local_validation_df[
            column_name
        ],
        errors="coerce",
    )

    local_numeric_validity_records.append(
        {
            "column": column_name,
            "row_count": len(numeric_values),
            "missing_or_unparseable_count": int(
                numeric_values.isna().sum()
            ),
            "nonfinite_count": int(
                (
                    numeric_values.notna()
                    & ~np.isfinite(numeric_values)
                ).sum()
            ),
            "nonpositive_count": int(
                (
                    numeric_values.notna()
                    & (numeric_values <= 0)
                ).sum()
            ),
            "minimum": numeric_values.min(),
            "maximum": numeric_values.max(),
        }
    )


local_numeric_validity_df = pd.DataFrame(
    local_numeric_validity_records
)


display(local_numeric_validity_df)

,column,row_count,missing_or_unparseable_count,nonfinite_count,nonpositive_count,minimum,maximum
0,frequency,233,0,0,0,4.116248e+01,4.409920e+02
1,bandwidth,233,0,0,0,5.859375e+07,2.000000e+09
2,sensitivity_10kms,233,0,0,0,4.375137e-01,2.883250e+01
3,cont_sensitivity_bandwidth,233,0,0,0,9.178503e-03,1.053364e+00
4,spatial_resolution,233,0,0,0,2.414189e-02,5.342102e+01
5,t_min,233,0,0,0,5.583188e+04,6.116462e+04
6,t_max,233,0,0,0,5.588486e+04,6.124735e+04


In [72]:
combined_frequency_support_text = (
    combined_local_validation_df[
        "frequency_support"
    ].map(
        normalize_frequency_support_text
    )
)


print(
    "Local rows with missing or blank "
    "frequency_support:",
    combined_frequency_support_text.isna().sum(),
)

Local rows with missing or blank frequency_support: 0


## Step 18 — Compare Source Contexts Across ASDMs by Position

Exact source-label equality found no source context repeated across multiple
ASDM executions. Because submitted source names are not reliable physical
identifiers, this experiment compares the reference coordinates and spatial
footprints of source contexts within the same Member OUS.

Small coordinate separation identifies candidates for further inspection but
does not by itself prove identical physical targets, especially for mosaics.

In [73]:
from itertools import combinations

from astropy.coordinates import SkyCoord


position_context_df = (
    additional_multi_asdm_df.copy()
)


position_context_df[
    "s_region_text"
] = position_context_df[
    "s_region"
].map(
    lambda value: (
        None
        if pd.isna(value)
        else str(value).strip()
    )
)


source_execution_context_df = (
    position_context_df
    .groupby(
        [
            "member_ous_uid",
            "asdm_uid",
            "obs_id_source",
        ],
        dropna=False,
    )
    .agg(
        archive_rows=("obs_id", "size"),
        spw_count=("spw_identifier", "nunique"),
        reference_ra_deg=("s_ra", "median"),
        reference_dec_deg=("s_dec", "median"),
        coordinate_count=("s_ra", "nunique"),
        spatial_footprint_count=(
            "s_region_text",
            "nunique",
        ),
        representative_footprint=(
            "s_region_text",
            "first",
        ),
        exact_signature_count=(
            "frequency_support_text",
            "nunique",
        ),
        representative_signature=(
            "frequency_support_text",
            "first",
        ),
        mosaic_state=("is_mosaic", "first"),
    )
    .reset_index()
)


display(source_execution_context_df)

,member_ous_uid,asdm_uid,obs_id_source,archive_rows,spw_count,reference_ra_deg,reference_dec_deg,coordinate_count,spatial_footprint_count,representative_footprint,exact_signature_count,representative_signature,mosaic_state
0,uid://A001/X15a9/X9f9,uid://A002/Xfe3986/X6550,Fil_18488_1,4,4,281.892254,-3.251909,1,1,Polygon ICRS 281.893285 -3.260064 281.890206 -3.259870 281.887414 -3.258559 281.885302 -3.256313 281.884423 -3.254449 281.884085 -3.252939 281.884085 -3.250878 281.884424 -3.24...,1,"[90.61..90.67GHz,35.28kHz,17.9mJy/beam@10km/s,3.9mJy/beam@native, XX YY] U [93.12..93.18GHz,35.28kHz,16.6mJy/beam@10km/s,3.7mJy/beam@native, XX YY] U [102.22..104.22GHz,1128.91...",F
1,uid://A001/X15a9/X9f9,uid://A002/Xfe3986/X6550,Fil_18488_2,4,4,281.849784,-3.223411,1,1,Polygon ICRS 281.851327 -3.231485 281.848242 -3.231485 281.845373 -3.230351 281.843124 -3.228243 281.842335 -3.226911 281.841697 -3.224951 281.841697 -3.221871 281.842335 -3.21...,1,"[90.61..90.67GHz,35.28kHz,18mJy/beam@10km/s,4mJy/beam@native, XX YY] U [93.12..93.18GHz,35.28kHz,16.7mJy/beam@10km/s,3.7mJy/beam@native, XX YY] U [102.22..104.22GHz,1128.91kHz,...",F
2,uid://A001/X15a9/X9f9,uid://A002/Xfe3986/X6550,Fil_21965_1,4,4,280.714020,-4.045874,1,1,Polygon ICRS 280.721476 -4.049374 280.719661 -4.051866 280.717054 -4.053517 280.714020 -4.054094 280.712476 -4.053949 280.710512 -4.053312 280.708013 -4.051501 280.706799 -4.04...,1,"[90.61..90.67GHz,35.28kHz,17.2mJy/beam@10km/s,3.8mJy/beam@native, XX YY] U [93.12..93.18GHz,35.28kHz,16.1mJy/beam@10km/s,3.6mJy/beam@native, XX YY] U [102.22..104.22GHz,1128.91...",F
3,uid://A001/X15a9/X9f9,uid://A002/Xfe62c1/Xb622,Fil_21965_2,4,4,280.714701,-3.998363,1,1,Polygon ICRS 280.715733 -4.006518 280.712651 -4.006324 280.709857 -4.005013 280.707743 -4.002767 280.706864 -4.000903 280.706526 -3.999393 280.706720 -3.996319 280.707480 -3.99...,1,"[90.61..90.67GHz,35.28kHz,18.2mJy/beam@10km/s,4mJy/beam@native, XX YY] U [93.12..93.18GHz,35.28kHz,16.7mJy/beam@10km/s,3.7mJy/beam@native, XX YY] U [102.22..104.22GHz,1128.91kH...",F
4,uid://A001/X15a9/X9f9,uid://A002/Xfe62c1/Xb622,Fil_21965_3,4,4,280.660972,-4.034593,1,1,Polygon ICRS 280.662005 -4.042748 280.658923 -4.042555 280.656129 -4.041243 280.654015 -4.038998 280.653135 -4.037133 280.652797 -4.035623 280.652748 -4.034077 280.653135 -4.03...,1,"[90.61..90.67GHz,35.28kHz,21.1mJy/beam@10km/s,4.6mJy/beam@native, XX YY] U [93.12..93.18GHz,35.28kHz,19.1mJy/beam@10km/s,4.3mJy/beam@native, XX YY] U [102.22..104.22GHz,1128.91...",F
5,uid://A001/X3577/X1df,uid://A002/X1026621/X18e5,RGALX320m01MPAID332386,1,1,320.059417,-0.567255,1,1,Circle ICRS 320.059417 -0.567255 0.015445,1,"[89.26..90.26GHz,564.45kHz,11.2mJy/beam@10km/s,615uJy/beam@native, XX YY]",F
6,uid://A001/X3577/X1df,uid://A002/X10afdb5/X6675,RGALX323m01MPAID333309,1,1,323.419279,-0.643250,1,1,Circle ICRS 323.419279 -0.643250 0.014569,1,"[94.66..95.66GHz,564.45kHz,5.7mJy/beam@10km/s,323.8uJy/beam@native, XX YY]",F
7,uid://A001/X3621/X233d,uid://A002/X1104ab5/X1a785,MCG-01.05.031,4,4,26.356083,-3.827111,1,1,Polygon ICRS 26.356367 -3.822617 26.358005 -3.823036 26.359373 -3.824028 26.360038 -3.824942 26.360376 -3.825719 26.360588 -3.827394 26.360455 -3.828231 26.360038 -3.829281 26....,1,"[177.79..179.38GHz,564.45kHz,1.5mJy/beam@10km/s,120uJy/beam@native, XX YY] U [178.68..179.61GHz,564.45kHz,1.5mJy/beam@10km/s,120uJy/beam@native, XX YY] U [179.57..180.51GHz,564...",F
8,uid://A001/X3621/X233d,uid://A002/X118bf6a/Xfff9,NGC0788,4,4,30.276875,-6.815528,1,1,Circle ICRS 30.276875 -6.815528 0.004483,1,"[178.60..179.54GHz,564.45kHz,1.1mJy/beam@10km/s,88.3uJy/beam@native, XX YY] U [179.49..180.43GHz,564.45kHz,1.1mJy/beam@10km/s,88.4uJy/beam@native, XX YY] U [180.39..181.32GHz,5...",F
9,uid://A001/X37fb/X35d,uid://A002/X127a8ae/X1fa31,PSR_J1745-2900,4,4,266.417359,-29.008304,1,1,Polygon ICRS 266.437285 -29.002641 266.438270 -29.009453 266.436319 -29.016105 266.431704 -29.021661 266.427454 -29.024361 266.423835 -29.025731 266.419986 -29.026484 266.41473..

In [74]:
context_pair_records = []


for member_uid, context_group in (
    source_execution_context_df.groupby(
        "member_ous_uid"
    )
):
    context_rows = list(
        context_group.itertuples(index=False)
    )

    for context_a, context_b in combinations(
        context_rows,
        2,
    ):
        coordinate_a = SkyCoord(
            ra=context_a.reference_ra_deg * u.deg,
            dec=context_a.reference_dec_deg * u.deg,
            frame="icrs",
        )

        coordinate_b = SkyCoord(
            ra=context_b.reference_ra_deg * u.deg,
            dec=context_b.reference_dec_deg * u.deg,
            frame="icrs",
        )

        separation_arcsec = (
            coordinate_a
            .separation(coordinate_b)
            .arcsec
        )

        context_pair_records.append(
            {
                "member_ous_uid": member_uid,
                "asdm_uid_a": context_a.asdm_uid,
                "source_a": context_a.obs_id_source,
                "asdm_uid_b": context_b.asdm_uid,
                "source_b": context_b.obs_id_source,
                "separation_arcsec": (
                    separation_arcsec
                ),
                "same_raw_source_label": (
                    context_a.obs_id_source
                    == context_b.obs_id_source
                ),
                "same_spatial_footprint": (
                    context_a.representative_footprint
                    == context_b.representative_footprint
                ),
                "same_exact_signature": (
                    context_a.representative_signature
                    == context_b.representative_signature
                ),
                "mosaic_state_a": (
                    context_a.mosaic_state
                ),
                "mosaic_state_b": (
                    context_b.mosaic_state
                ),
            }
        )


context_position_comparison_df = pd.DataFrame(
    context_pair_records
)


display(
    context_position_comparison_df
    .sort_values("separation_arcsec")
    .head(50)
)

,member_ous_uid,asdm_uid_a,source_a,asdm_uid_b,source_b,separation_arcsec,same_raw_source_label,same_spatial_footprint,same_exact_signature,mosaic_state_a,mosaic_state_b
7,uid://A001/X15a9/X9f9,uid://A002/Xfe3986/X6550,Fil_21965_1,uid://A002/Xfe62c1/Xb622,Fil_21965_2,171.059089,False,False,False,F,F
0,uid://A001/X15a9/X9f9,uid://A002/Xfe3986/X6550,Fil_18488_1,uid://A002/Xfe3986/X6550,Fil_18488_2,183.916738,False,False,False,F,F
8,uid://A001/X15a9/X9f9,uid://A002/Xfe3986/X6550,Fil_21965_1,uid://A002/Xfe62c1/Xb622,Fil_21965_3,194.779149,False,False,False,F,F
9,uid://A001/X15a9/X9f9,uid://A002/Xfe62c1/Xb622,Fil_21965_2,uid://A002/Xfe62c1/Xb622,Fil_21965_3,232.895815,False,False,False,F,F
5,uid://A001/X15a9/X9f9,uid://A002/Xfe3986/X6550,Fil_18488_2,uid://A002/Xfe62c1/Xb622,Fil_21965_2,4941.103329,False,False,False,F,F
2,uid://A001/X15a9/X9f9,uid://A002/Xfe3986/X6550,Fil_18488_1,uid://A002/Xfe62c1/Xb622,Fil_21965_2,5011.972390,False,False,False,F,F
4,uid://A001/X15a9/X9f9,uid://A002/Xfe3986/X6550,Fil_18488_2,uid://A002/Xfe3986/X6550,Fil_21965_1,5041.542337,False,False,False,F,F
1,uid://A001/X15a9/X9f9,uid://A002/Xfe3986/X6550,Fil_18488_1,uid://A002/Xfe3986/X6550,Fil_21965_1,5107.650483,False,False,False,F,F
6,uid://A001/X15a9/X9f9,uid://A002/Xfe3986/X6550,Fil_18488_2,uid://A002/Xfe62c1/Xb622,Fil_21965_3,5173.996514,False,False,False,F,F
3,uid://A001/X15a9/X9f9,uid://A002/Xfe3986/X6550,Fil_18488_1,uid://A002/Xfe62c1/Xb622,Fil_21965_3,5244.773037,False,False,False,F,F


In [75]:
close_context_candidates_df = (
    context_position_comparison_df[
        context_position_comparison_df[
            "separation_arcsec"
        ]
        < 60
    ]
    .sort_values("separation_arcsec")
    .reset_index(drop=True)
)


print(
    "Context pairs separated by less than "
    "60 arcsec:",
    len(close_context_candidates_df),
)


display(close_context_candidates_df)

Context pairs separated by less than 60 arcsec: 0


,member_ous_uid,asdm_uid_a,source_a,asdm_uid_b,source_b,separation_arcsec,same_raw_source_label,same_spatial_footprint,same_exact_signature,mosaic_state_a,mosaic_state_b


## Final Conclusions

1. `frequency_support` is a composite spectral-configuration description,
   rather than one scalar frequency value.

2. In the tested records, each bracketed component contained a frequency
   interval, spectral resolution, sensitivity at 10 km/s, sensitivity at native
   resolution, and polarization products.

3. The main purposive sample contained 18 Member OUS datasets, 171 Archive
   rows, and 95 parsed frequency-support components. All 95 components were
   parsed successfully.

4. Frequency-support component count matched logical-SPW count for all 18
   Member OUS datasets. All 18 datasets also formed complete source × SPW
   grids.

5. A one-to-one numerical assignment matched all 95 logical SPWs to parsed
   support components. Every row-level reference frequency fell inside its
   assigned parsed interval.

6. Row-level `frequency`, Archive `bandwidth`, parsed interval centre, and
   parsed interval width are related but not interchangeable. The maximum
   observed absolute centre difference was approximately 4.705 MHz, and the
   maximum bandwidth-versus-interval-width difference was 30 MHz in the main
   sample.

7. Exact `frequency_support` string equality is too strict for frequency
   coverage comparison. The experiments found both:
   - identical spectral geometry with different sensitivity metadata;
   - different spectral geometries within the same Member OUS.

8. The parsed `@10km/s` sensitivity closely matched the row-level
   `sensitivity_10kms` value, but small differences remained. Both raw
   representations must be preserved. Component-level native sensitivity must
   not be treated as equivalent to `cont_sensitivity_bandwidth`.

9. The tested grammar was stable across purposive samples from ALMA Bands 3–10,
   but this is not an Archive-wide grammar guarantee. Raw strings, parse status,
   validation issues, and parser version must be preserved.

10. The ownership level of `frequency_support` is not fully resolved. The
    current model should associate the raw signature conservatively with a
    Source–Execution Context rather than assuming one signature per Member OUS,
    Source Context, or ASDM.

11. Frequency-equivalence tolerances are duplication-policy decisions and are
    not determined by the numerical differences observed in this notebook.